# NB06 V2 — Native / Audio / Image Diagnostic Integration

## Purpose

This notebook is a clean diagnostic integration notebook for the Mindray TE7 DopplerLab batch.

It does **not** replace the video-based Doppler feature extraction pipeline.

It integrates:

- native metadata calibration checks,
- native / AVI / audio duration agreement,
- manual native-to-NB04 mapping,
- audio / image / native availability,
- documentation of failed native beat-timing validation,
- optional experimental native coarse-HR QC status,
- final recording-level `recommended_use` labels.

## Scope

### Confirmed and usable

- `DcmRegionPara.txt` can provide PW spectral ROI, baseline, time scale, and velocity scale for AVI/image calibration.
- `PAGE_RATE_HZ = 500` is the native PW page-rate reference.
- `n_pages / 500` is the primary native duration estimate.
- Native metadata can reduce manual ROI/baseline/time-scale setup.

### Experimental only

- Native `field_b` / `pair_range` / PCA-style signals may contain coarse cardiac activity.
- Native coarse HR from ACF may be useful as a QC sidecar, but not as a final feature.

### Failed / not integrated

- Native pair-range individual beat timing failed validation.
- Native binary is not used for PSV, EDV, RI, PI, VTI, SV, CO, BP, or clinical diagnosis.
- `field_a` codebook, FeParam, PHYSIO, and BC partition exploration remain future research / documentation only.

## Output

Primary output:

```text
reports/nb06_v2/nb06_v2_recording_diagnostic.csv

## Imports &  Setup

In [1]:
from pathlib import Path
import json
import csv
import re
import subprocess
import warnings
import numpy as np
import pandas as pd

# NB06 V2 — setup and project paths

PAGE_SIZE_BYTES = 1296
HEADER_SIZE_BYTES = 16
RECORDS_PER_PAGE = 160
PAGE_RATE_HZ = 500.0

NATIVE_BEAT_TIMING_STATUS = "failed_validation"

In [2]:
def resolve_project_root(candidate_roots=None):
    """
    Resolve the DopplerLab project root.

    The project has been used from more than one Windows path during development,
    so this helper checks known candidate locations and returns the first valid one.

    A valid DopplerLab root should contain:
    - ultrasound_recordings/
    - or feature_exports/
    - or scope/

    Parameters
    ----------
    candidate_roots : list[str or pathlib.Path] or None
        Candidate project roots to test.

    Returns
    -------
    pathlib.Path
        Resolved project root.

    Raises
    ------
    FileNotFoundError
        If none of the candidate roots exists.
    """
    if candidate_roots is None:
        candidate_roots = [
            Path(r"E:\DopplerLab"),
            Path(r"D:\code\DopplerLab"),
            Path.cwd(),
            Path.cwd().parent,
        ]

    checked = []

    for root in candidate_roots:
        root = Path(root)
        checked.append(str(root))

        if not root.exists():
            continue

        expected_any = [
            root / "ultrasound_recordings",
            root / "feature_exports",
            root / "scope",
        ]

        if any(path.exists() for path in expected_any):
            return root

    raise FileNotFoundError("Could not resolve DopplerLab project root. Checked:\n" + "\n".join(checked))



def build_nb06_v2_paths(project_root):
    """
    Build all paths used by NB06 V2.

    Parameters
    ----------
    project_root : pathlib.Path
        DopplerLab project root.

    Returns
    -------
    dict
        Dictionary with named project paths.
    """
    project_root = Path(project_root)

    paths = {
        "PROJECT_ROOT": project_root,

        # Input data
        "ULTRASOUND_RECORDINGS_DIR": project_root / "ultrasound_recordings",
        "NATIVE_BATCH_DIR": project_root / "ultrasound_recordings" / "batch_2026_06_13_native",
        "NB04_V2_DIR": project_root / "feature_exports" / "nb04_v2_batch_2026_06_13",

        # Existing reports from previous native validation / Claude scope runs
        "REPORTS_DIR": project_root / "reports",
        "DOCS_DIR": project_root / "docs",
        "FIGURES_DIR": project_root / "figures",
        "SCOPE_DIR": project_root / "scope",

        # Important existing CSVs
        "NATIVE_TO_NB04_TEMPLATE_CSV": project_root / "reports" / "native_to_nb04_manual_match_template.csv",
        "NATIVE_TIMING_VALIDATION_CSV": project_root / "reports" / "native_timing_validation.csv",
        "NATIVE_PAIR_RANGE_VALIDATION_CSV": project_root / "reports" / "native_pair_range_validation.csv",
        "NB04_REGISTRY_CSV": project_root / "feature_exports" / "nb04_v2_batch_2026_06_13" / "nb04_v2_accepted_recording_registry.csv",
        "NB04_MORPHOLOGY_CSV": project_root / "feature_exports" / "nb04_v2_batch_2026_06_13" / "nb04_v2_complete_morphology_candidates.csv",

        # Optional scope outputs
        "SCOPE01_NATIVE_SIGNAL_SUMMARY_CSV": project_root / "scope" / "01_native_signal_qc_family" / "reports" / "native_signal_qc_family_summary.csv",
        "SCOPE01_NATIVE_SIGNAL_REFINEMENT_CSV": project_root / "scope" / "01_native_signal_qc_family" / "reports" / "native_signal_qc_family_refinement.csv",
        "SCOPE02_BMODE_FEASIBILITY_CSV": project_root / "scope" / "02_upper_bmode_artery_tracking_feasibility" / "reports" / "bmode_artery_tracking_feasibility.csv",
        "SCOPE03_FIELD_A_SLOTS_CSV": project_root / "scope" / "03_field_a_codebook_forensics" / "reports" / "field_a_codebook_slots.csv",
        "SCOPE04_AUXILIARY_DOC": project_root / "scope" / "04_physio_feparam_header_forensics" / "docs" / "NATIVE_AUXILIARY_FORENSICS.md",

        # NB06 V2 outputs
        "NB06_V2_REPORTS_DIR": project_root / "reports" / "nb06_v2",
        "NB06_V2_FIGURES_DIR": project_root / "figures" / "nb06_v2",
        "NB06_V2_DOCS_DIR": project_root / "docs" / "nb06_v2",
    }

    return paths


def ensure_nb06_v2_output_dirs(paths):
    """
    Create NB06 V2 output directories.

    Parameters
    ----------
    paths : dict
        Paths dictionary returned by build_nb06_v2_paths().
    """

    for key in [
        "NB06_V2_REPORTS_DIR",
        "NB06_V2_FIGURES_DIR",
        "NB06_V2_DOCS_DIR",
    ]:
        paths[key].mkdir(parents=True, exist_ok=True)


def path_status_table(paths):
    """
    Build a path status table for NB06 V2 dependencies.

    Parameters
    ----------
    paths : dict
        Paths dictionary returned by build_nb06_v2_paths().

    Returns
    -------
    pandas.DataFrame
        One row per path with existence and role.
    """
    required_keys = {
        "PROJECT_ROOT",
        "NATIVE_BATCH_DIR",
        "NB04_V2_DIR",
        "NB04_REGISTRY_CSV",
        "NB04_MORPHOLOGY_CSV",
    }

    recommended_keys = {
        "NATIVE_TO_NB04_TEMPLATE_CSV",
        "NATIVE_TIMING_VALIDATION_CSV",
        "NATIVE_PAIR_RANGE_VALIDATION_CSV",
    }

    optional_keys = {
        "SCOPE01_NATIVE_SIGNAL_SUMMARY_CSV",
        "SCOPE01_NATIVE_SIGNAL_REFINEMENT_CSV",
        "SCOPE02_BMODE_FEASIBILITY_CSV",
        "SCOPE03_FIELD_A_SLOTS_CSV",
        "SCOPE04_AUXILIARY_DOC",
    }

    output_keys = {
        "NB06_V2_REPORTS_DIR",
        "NB06_V2_FIGURES_DIR",
        "NB06_V2_DOCS_DIR",
    }

    rows = []

    for key, path in paths.items():
        if key in required_keys:
            role = "required"
        elif key in recommended_keys:
            role = "recommended"
        elif key in optional_keys:
            role = "optional_scope_output"
        elif key in output_keys:
            role = "output"
        else:
            role = "support"

        path = Path(path)

        rows.append(
            {
                "key": key,
                "role": role,
                "exists": path.exists(),
                "is_dir": path.is_dir() if path.exists() else False,
                "path": str(path),
            }
        )

    status_df = pd.DataFrame(rows).sort_values(["role", "key"]).reset_index(drop=True)

    return status_df


def print_nb06_v2_setup_summary(project_root, paths, status_df):
    """
    Print a compact setup summary.

    Parameters
    ----------
    project_root : pathlib.Path
        Resolved project root.

    paths : dict
        Path dictionary.

    status_df : pandas.DataFrame
        Path status table.
    """
    print("NB06 V2 setup")
    print("=" * 80)
    print(f"PROJECT_ROOT: {project_root}")
    print()

    missing_required = status_df[(status_df["role"] == "required") & (~status_df["exists"])]
    missing_recommended = status_df[(status_df["role"] == "recommended") & (~status_df["exists"])]

    print(f"Required paths missing: {len(missing_required)}")
    print(f"Recommended paths missing: {len(missing_recommended)}")
    print()

    if len(missing_required) > 0:
        print("Missing required paths:")
        for _, row in missing_required.iterrows():
            print(f"  - {row['key']}: {row['path']}")
        print()

    if len(missing_recommended) > 0:
        print("Missing recommended paths:")
        for _, row in missing_recommended.iterrows():
            print(f"  - {row['key']}: {row['path']}")
        print()

    print("Output directories:")
    for key in ["NB06_V2_REPORTS_DIR", "NB06_V2_FIGURES_DIR", "NB06_V2_DOCS_DIR"]:
        print(f"  - {key}: {paths[key]}")


In [3]:
PROJECT_ROOT = resolve_project_root()
PATHS = build_nb06_v2_paths(PROJECT_ROOT)
ensure_nb06_v2_output_dirs(PATHS)

setup_status_df = path_status_table(PATHS)
print_nb06_v2_setup_summary(PROJECT_ROOT, PATHS, setup_status_df)

setup_status_df

NB06 V2 setup
PROJECT_ROOT: E:\DopplerLab

Required paths missing: 0
Recommended paths missing: 0

Output directories:
  - NB06_V2_REPORTS_DIR: E:\DopplerLab\reports\nb06_v2
  - NB06_V2_FIGURES_DIR: E:\DopplerLab\figures\nb06_v2
  - NB06_V2_DOCS_DIR: E:\DopplerLab\docs\nb06_v2


,key,role,exists,is_dir,path
0,SCOPE01_NATIVE_SIGNAL_REFINEMENT_CSV,optional_scope_output,True,False,E:\DopplerLab\scope\01_native_signal_qc_family...
1,SCOPE01_NATIVE_SIGNAL_SUMMARY_CSV,optional_scope_output,True,False,E:\DopplerLab\scope\01_native_signal_qc_family...
2,SCOPE02_BMODE_FEASIBILITY_CSV,optional_scope_output,True,False,E:\DopplerLab\scope\02_upper_bmode_artery_trac...
3,SCOPE03_FIELD_A_SLOTS_CSV,optional_scope_output,True,False,E:\DopplerLab\scope\03_field_a_codebook_forens...
4,SCOPE04_AUXILIARY_DOC,optional_scope_output,True,False,E:\DopplerLab\scope\04_physio_feparam_header_f...
5,NB06_V2_DOCS_DIR,output,True,True,E:\DopplerLab\docs\nb06_v2
6,NB06_V2_FIGURES_DIR,output,True,True,E:\DopplerLab\figures\nb06_v2
7,NB06_V2_REPORTS_DIR,output,True,True,E:\DopplerLab\reports\nb06_v2
8,NATIVE_PAIR_RANGE_VALIDATION_CSV,recommended,True,False,E:\DopplerLab\reports\native_pair_range_valida...
9,NATIVE_TIMING_VALIDATION_CSV,recommended,True,False,E:\DopplerLab\reports\native_timing_validation...


## Native recording inventory

This section builds the base recording inventory for NB06 V2.

It checks, per native recording:

- native folder availability,
- linked AVI availability,
- PW binary availability,
- metadata files availability,
- PW binary page count,
- native duration from `n_pages / 500`.

This is the foundation for the later diagnostic table.
No Doppler features are extracted here.

In [4]:
def find_first_existing_file(folder, candidate_names=None, glob_patterns=None):
    """
    Find the first existing file in a folder using exact candidate names
    and/or glob patterns.

    Parameters
    ----------
    folder : pathlib.Path
        Folder to search.

    candidate_names : list[str] or None
        Exact filenames to check first.

    glob_patterns : list[str] or None
        Glob patterns to check if exact filenames are not found.

    Returns
    -------
    pathlib.Path or None
        First matching file, or None if no match is found.
    """
    folder = Path(folder)

    if not folder.exists():
        return None

    if candidate_names is not None:
        for name in candidate_names:
            candidate = folder / name
            if candidate.exists() and candidate.is_file():
                return candidate

    if glob_patterns is not None:
        for pattern in glob_patterns:
            matches = sorted(folder.glob(pattern))
            matches = [path for path in matches if path.is_file()]
            if len(matches) > 0:
                return matches[0]

    return None


def safe_file_size(path):
    """
    Return file size in bytes, or None if path is missing.

    Parameters
    ----------
    path : pathlib.Path or None
        File path.

    Returns
    -------
    int or None
        File size in bytes.
    """
    if path is None:
        return None

    path = Path(path)

    if not path.exists() or not path.is_file():
        return None

    return int(path.stat().st_size)


def compute_pw_page_count_from_size(file_size_bytes):
    """
    Compute PW native page count from file size.

    Parameters
    ----------
    file_size_bytes : int or None
        Size of PW_CinePartition0.bin.

    Returns
    -------
    dict
        Page count, duration, and remainder information.
    """
    if file_size_bytes is None:
        return {
            "pw_n_pages": None,
            "duration_pages_s": None,
            "pw_size_remainder_bytes": None,
            "pw_size_multiple_of_page": False,
        }

    n_pages = file_size_bytes // PAGE_SIZE_BYTES
    remainder = file_size_bytes % PAGE_SIZE_BYTES

    return {
        "pw_n_pages": int(n_pages),
        "duration_pages_s": round(float(n_pages / PAGE_RATE_HZ), 3),
        "pw_size_remainder_bytes": int(remainder),
        "pw_size_multiple_of_page": bool(remainder == 0),
    }


def discover_native_recording_folder(recording_folder):
    """
    Discover important files inside one native recording folder.

    Expected folder shape:

        recording_id/
            native/
                PW_CinePartition0.bin
                DcmRegionPara.txt
                app.xml
                VirtualMachine.txt
                VirtualMachine.bin
                BackEndSystemInfo.txt
                BackEndSystem.txt
                BC_CinePartition1.bin
            linked_avi/
                recording_id.avi

    Parameters
    ----------
    recording_folder : pathlib.Path
        Native recording folder.

    Returns
    -------
    dict
        One inventory row.
    """
    recording_folder = Path(recording_folder)
    recording_id = recording_folder.name

    native_subdir = recording_folder / "native"
    linked_avi_dir = recording_folder / "linked_avi"

    pw_bin_path = find_first_existing_file(
        native_subdir,
        candidate_names=["PW_CinePartition0.bin"],
        glob_patterns=["PW_CinePartition*.bin"],
    )

    bc_bin_path = find_first_existing_file(
        native_subdir,
        candidate_names=["BC_CinePartition1.bin"],
        glob_patterns=["BC_CinePartition*.bin"],
    )

    linked_avi_path = find_first_existing_file(
        linked_avi_dir,
        candidate_names=[f"{recording_id}.avi"],
        glob_patterns=["*.avi", "*.mp4"],
    )

    dcm_region_para_path = find_first_existing_file(
        native_subdir,
        candidate_names=["DcmRegionPara.txt"],
    )

    app_xml_path = find_first_existing_file(
        native_subdir,
        candidate_names=["app.xml"],
    )

    virtual_machine_txt_path = find_first_existing_file(
        native_subdir,
        candidate_names=["VirtualMachine.txt"],
    )

    virtual_machine_bin_path = find_first_existing_file(
        native_subdir,
        candidate_names=["VirtualMachine.bin"],
    )

    backend_system_info_path = find_first_existing_file(
        native_subdir,
        candidate_names=["BackEndSystemInfo.txt"],
    )

    backend_system_txt_path = find_first_existing_file(
        native_subdir,
        candidate_names=["BackEndSystem.txt"],
    )

    pw_bin_size_bytes = safe_file_size(pw_bin_path)
    bc_bin_size_bytes = safe_file_size(bc_bin_path)
    linked_avi_size_bytes = safe_file_size(linked_avi_path)

    pw_page_info = compute_pw_page_count_from_size(pw_bin_size_bytes)

    row = {
        "recording_id": recording_id,
        "recording_folder": str(recording_folder),
        "native_subdir": str(native_subdir),
        "linked_avi_dir": str(linked_avi_dir),

        "has_native_subdir": native_subdir.exists(),
        "has_linked_avi_dir": linked_avi_dir.exists(),

        "pw_bin_path": str(pw_bin_path) if pw_bin_path else "",
        "bc_bin_path": str(bc_bin_path) if bc_bin_path else "",
        "linked_avi_path": str(linked_avi_path) if linked_avi_path else "",
        "dcm_region_para_path": str(dcm_region_para_path) if dcm_region_para_path else "",
        "app_xml_path": str(app_xml_path) if app_xml_path else "",
        "virtual_machine_txt_path": str(virtual_machine_txt_path) if virtual_machine_txt_path else "",
        "virtual_machine_bin_path": str(virtual_machine_bin_path) if virtual_machine_bin_path else "",
        "backend_system_info_path": str(backend_system_info_path) if backend_system_info_path else "",
        "backend_system_txt_path": str(backend_system_txt_path) if backend_system_txt_path else "",

        "has_pw_bin": pw_bin_path is not None,
        "has_bc_bin": bc_bin_path is not None,
        "has_linked_avi": linked_avi_path is not None,
        "has_dcm_region_para": dcm_region_para_path is not None,
        "has_app_xml": app_xml_path is not None,
        "has_virtual_machine_txt": virtual_machine_txt_path is not None,
        "has_virtual_machine_bin": virtual_machine_bin_path is not None,
        "has_backend_system_info": backend_system_info_path is not None,
        "has_backend_system_txt": backend_system_txt_path is not None,

        "pw_bin_size_bytes": pw_bin_size_bytes,
        "bc_bin_size_bytes": bc_bin_size_bytes,
        "linked_avi_size_bytes": linked_avi_size_bytes,
    }

    row.update(pw_page_info)

    return row


def build_native_recording_inventory(native_batch_dir):
    """
    Build native recording inventory for the whole batch.

    Parameters
    ----------
    native_batch_dir : pathlib.Path
        Native batch folder.

    Returns
    -------
    pandas.DataFrame
        Recording-level inventory table.
    """
    native_batch_dir = Path(native_batch_dir)

    if not native_batch_dir.exists():
        raise FileNotFoundError(f"Native batch directory not found: {native_batch_dir}")

    recording_folders = [
        path for path in sorted(native_batch_dir.iterdir())
        if path.is_dir()
    ]

    rows = []

    for recording_folder in recording_folders:
        row = discover_native_recording_folder(recording_folder)
        rows.append(row)

    inventory_df = pd.DataFrame(rows)

    if len(inventory_df) == 0:
        return inventory_df

    inventory_df = inventory_df.sort_values("recording_id").reset_index(drop=True)

    return inventory_df


def summarize_native_recording_inventory(inventory_df):
    """
    Print a compact inventory summary.

    Parameters
    ----------
    inventory_df : pandas.DataFrame
        Recording-level inventory table.
    """
    print("Native recording inventory")
    print("=" * 80)
    print(f"Recordings found: {len(inventory_df)}")

    if len(inventory_df) == 0:
        return

    check_cols = [
        "has_pw_bin",
        "has_linked_avi",
        "has_dcm_region_para",
        "has_app_xml",
        "has_virtual_machine_txt",
        "has_backend_system_info",
    ]

    print()
    for col in check_cols:
        if col in inventory_df.columns:
            print(f"{col}: {int(inventory_df[col].sum())}/{len(inventory_df)}")

    print()
    print("PW page count / duration summary:")

    if "pw_n_pages" in inventory_df.columns:
        display_cols = [
            "recording_id",
            "pw_n_pages",
            "duration_pages_s",
            "pw_size_remainder_bytes",
            "pw_size_multiple_of_page",
            "has_linked_avi",
            "has_dcm_region_para",
        ]

        display_cols = [col for col in display_cols if col in inventory_df.columns]
        display(inventory_df[display_cols])


def save_native_recording_inventory(inventory_df, output_dir):
    """
    Save native recording inventory to NB06 V2 reports directory.

    Parameters
    ----------
    inventory_df : pandas.DataFrame
        Recording-level inventory table.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    pathlib.Path
        Saved CSV path.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "nb06_v2_native_recording_inventory.csv"

    inventory_df.to_csv(output_path, index=False)

    print(f"Saved inventory CSV: {output_path}")

    return output_path



In [5]:
native_inventory_df = build_native_recording_inventory(PATHS["NATIVE_BATCH_DIR"])
summarize_native_recording_inventory(native_inventory_df)
native_inventory_csv_path = save_native_recording_inventory(native_inventory_df, PATHS["NB06_V2_REPORTS_DIR"],)
native_inventory_df

Native recording inventory
Recordings found: 10

has_pw_bin: 10/10
has_linked_avi: 10/10
has_dcm_region_para: 10/10
has_app_xml: 10/10
has_virtual_machine_txt: 10/10
has_backend_system_info: 10/10

PW page count / duration summary:


,recording_id,pw_n_pages,duration_pages_s,pw_size_remainder_bytes,pw_size_multiple_of_page,has_linked_avi,has_dcm_region_para
0,202606130411060002SMP,29615,59.230,0,True,True,True
1,202606130413540003SMP,23491,46.982,0,True,True,True
2,202606130417060004SMP,21232,42.464,0,True,True,True
3,202606130420260005SMP,22764,45.528,0,True,True,True
4,202606130422440006SMP,25626,51.252,0,True,True,True
5,202606130426500007SMP,18346,36.692,0,True,True,True
6,202606130430380008SMP,22186,44.372,0,True,True,True
7,202606130433280009SMP,15862,31.724,0,True,True,True
8,202606130434130010SMP,16916,33.832,0,True,True,True
9,202606130437480012SMP,22513,45.026,0,True,True,True


Saved inventory CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_native_recording_inventory.csv


,recording_id,recording_folder,native_subdir,linked_avi_dir,has_native_subdir,has_linked_avi_dir,pw_bin_path,bc_bin_path,linked_avi_path,dcm_region_para_path,...,has_virtual_machine_bin,has_backend_system_info,has_backend_system_txt,pw_bin_size_bytes,bc_bin_size_bytes,linked_avi_size_bytes,pw_n_pages,duration_pages_s,pw_size_remainder_bytes,pw_size_multiple_of_page
0,202606130411060002SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,38381040,174368,14816678,29615,59.230,0,True
1,202606130413540003SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,30444336,174368,12845344,23491,46.982,0,True
2,202606130417060004SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,27516672,174368,9287210,21232,42.464,0,True
3,202606130420260005SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,29502144,174368,15672588,22764,45.528,0,True
4,202606130422440006SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,33211296,174368,15822224,25626,51.252,0,True
5,202606130426500007SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,23776416,174368,12638752,18346,36.692,0,True
6,202606130430380008SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,28753056,174368,10053972,22186,44.372,0,True
7,202606130433280009SMP,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...,...,True,True,True,20557152,174368,13273528,15862,31.724,0,True
8,202606130434130010SMP,E:\DopplerLab\ultrasound_

## DcmRegionPara / PW calibration QC

This section parses `DcmRegionPara.txt` for each native recording.

Main goal:

- find the PW spectral Doppler region,
- extract ROI bounds,
- extract baseline / zero-velocity row,
- extract time scale from `PhyDeltaX`,
- extract velocity scale from `PhyDeltaY`,
- verify that metadata are internally consistent.

Important:

`DcmRegionPara.txt` metadata can be used to calibrate the linked AVI/image extraction.

The native binary itself is still **not** used for velocity-envelope extraction.

In [6]:
def parse_dcm_region_para_file(dcm_region_para_path):
    """
    Parse Mindray DcmRegionPara.txt into a structured dictionary.

    Parameters
    ----------
    dcm_region_para_path : pathlib.Path or str
        Path to DcmRegionPara.txt.

    Returns
    -------
    dict
        Parsed metadata with:
        - n_regions_declared
        - regions: dict of region_name -> parameter dict
        - parse_ok
        - error
    """
    dcm_region_para_path = Path(dcm_region_para_path)
    result = {
        "path": str(dcm_region_para_path),
        "parse_ok": False,
        "error": "",
        "n_regions_declared": None,
        "regions": {},
    }

    if not dcm_region_para_path.exists():
        result["error"] = "file_not_found"
        return result

    try:
        text = dcm_region_para_path.read_text(encoding="utf-8", errors="replace",)
    except Exception as exc:
        result["error"] = f"read_error: {exc}"
        return result

    n_regions_match = re.search(r"DcmRegionNum=(\d+)", text)
    if n_regions_match:
        result["n_regions_declared"] = int(n_regions_match.group(1))

    region_names = re.findall(r"DATA_TREE_BEGIN=(DcmRegion\d+)", text)

    for region_name in region_names:
        start_token = f"DATA_TREE_BEGIN={region_name}"
        end_token = f"DATA_TREE_END={region_name}"
        start_idx = text.find(start_token)
        end_idx = text.find(end_token, start_idx)

        if start_idx == -1 or end_idx == -1:
            continue

        block = text[start_idx:end_idx]
        params = {}

        for line in block.splitlines():
            match = re.match(r"\s*([^=\s]+)\s*=\s*(.+?)\s*$", line)

            if match is None:
                continue

            key = match.group(1).strip()
            value = match.group(2).strip()
            params[key] = value

        result["regions"][region_name] = params

    result["parse_ok"] = len(result["regions"]) > 0

    if not result["parse_ok"]:
        result["error"] = "no_regions_parsed"

    return result


def safe_float_from_params(params, key):
    """
    Safely read a float value from region params.

    Parameters
    ----------
    params : dict
        Region parameter dictionary.

    key : str
        Parameter key.

    Returns
    -------
    float or None
        Parsed float value, or None.
    """
    try:
        return float(params[key])
    except Exception:
        return None


def safe_int_from_params(params, key):
    """
    Safely read an integer value from region params.

    Parameters
    ----------
    params : dict
        Region parameter dictionary.

    key : str
        Parameter key.

    Returns
    -------
    int or None
        Parsed integer value, or None.
    """
    try:
        return int(float(params[key]))
    except Exception:
        return None


def classify_dcm_region(region_name, params):
    """
    Classify one DcmRegionPara region.

    This is conservative and descriptive.
    It does not infer clinical meaning.

    Parameters
    ----------
    region_name : str
        Region name, for example DcmRegion0.

    params : dict
        Region parameter dictionary.

    Returns
    -------
    str
        Region class label.
    """
    data_type = params.get("DataType", "")
    phy_units_x = params.get("PhyUnitsX", "")
    phy_units_y = params.get("PhyUnitsY", "")

    phy_delta_x = safe_float_from_params(params, "PhyDeltaX")
    phy_delta_y = safe_float_from_params(params, "PhyDeltaY")

    # PW spectral Doppler region:
    # DataType=3 and PhyUnitsY=7 were observed in validated metadata.
    if data_type == "3" and phy_units_y == "7":
        return "pw_spectral_region"

    # B-mode / spatial image region.
    # Keep this broad because exact Mindray labels may vary.
    if data_type in ("1", "0") and phy_delta_x is not None and phy_delta_y is not None:
        if abs(phy_delta_x) > 0 and abs(phy_delta_y) > 0:
            if abs(phy_delta_x) < 0.1 and abs(phy_delta_y) < 0.1:
                return "spatial_image_region"

    return "unknown_region"


def build_dcm_region_summary_rows(recording_id, parsed_dcm):
    """
    Build one row per DcmRegionPara region.

    Parameters
    ----------
    recording_id : str
        Native recording ID.

    parsed_dcm : dict
        Parsed DcmRegionPara dictionary.

    Returns
    -------
    list[dict]
        Region summary rows.
    """
    rows = []

    for region_name, params in parsed_dcm["regions"].items():
        region_class = classify_dcm_region(region_name, params)

        x0 = safe_int_from_params(params, "X0")
        x1 = safe_int_from_params(params, "X1")
        y0 = safe_int_from_params(params, "Y0")
        y1 = safe_int_from_params(params, "Y1")
        vir_y = safe_int_from_params(params, "VirY")
        phy_delta_x = safe_float_from_params(params, "PhyDeltaX")
        phy_delta_y = safe_float_from_params(params, "PhyDeltaY")

        row = {
            "recording_id": recording_id,
            "region_name": region_name,
            "region_class": region_class,

            "DataType": params.get("DataType", ""),
            "SpatialFormat": params.get("SpatialFormat", ""),
            "RegionFlags": params.get("RegionFlags", ""),
            "PhyUnitsX": params.get("PhyUnitsX", ""),
            "PhyUnitsY": params.get("PhyUnitsY", ""),

            "X0": x0,
            "X1": x1,
            "Y0": y0,
            "Y1": y1,
            "VirX": safe_int_from_params(params, "VirX"),
            "VirY": vir_y,

            "PhyDeltaX": phy_delta_x,
            "PhyDeltaY": phy_delta_y,

            "roi_width_px": (x1 - x0) if x0 is not None and x1 is not None else None,
            "roi_height_px": (y1 - y0) if y0 is not None and y1 is not None else None,

            "baseline_y_global_px": (y0 + vir_y) if y0 is not None and vir_y is not None else None,
        }

        rows.append(row)

    return rows


def select_pw_spectral_region(region_rows):
    """
    Select the PW spectral region from one recording's DcmRegionPara rows.

    Parameters
    ----------
    region_rows : list[dict]
        Region rows for one recording.

    Returns
    -------
    dict or None
        Selected PW region row, or None.
    """
    pw_candidates = [row for row in region_rows if row.get("region_class") == "pw_spectral_region"]

    if len(pw_candidates) == 1:
        return pw_candidates[0]
    if len(pw_candidates) > 1:
        # Prefer region with plausible Doppler spectral time scale.
        plausible = []

        for row in pw_candidates:
            phy_delta_x = row.get("PhyDeltaX")
            phy_delta_y = row.get("PhyDeltaY")

            if phy_delta_x is None or phy_delta_y is None:
                continue

            if 0.001 <= abs(phy_delta_x) <= 0.02 and 0.01 <= abs(phy_delta_y) <= 1.0:
                plausible.append(row)

        if len(plausible) > 0:
            return plausible[0]

        return pw_candidates[0]

    # Fallback: choose a region that looks like the known PW region:
    # time-like X scale and velocity-like Y scale.
    fallback_candidates = []

    for row in region_rows:
        phy_delta_x = row.get("PhyDeltaX")
        phy_delta_y = row.get("PhyDeltaY")
        roi_height = row.get("roi_height_px")
        baseline = row.get("baseline_y_global_px")

        if phy_delta_x is None or phy_delta_y is None:
            continue

        looks_time_like = 0.001 <= abs(phy_delta_x) <= 0.02
        looks_velocity_like = 0.01 <= abs(phy_delta_y) <= 1.0
        has_reasonable_height = roi_height is not None and roi_height > 100
        has_baseline = baseline is not None

        if looks_time_like and looks_velocity_like and has_reasonable_height and has_baseline:
            fallback_candidates.append(row)

    if len(fallback_candidates) > 0:
        return fallback_candidates[0]

    return None


def evaluate_pw_calibration_qc(pw_row):
    """
    Evaluate QC of one selected PW spectral region.

    Parameters
    ----------
    pw_row : dict or None
        Selected PW region row.

    Returns
    -------
    dict
        QC fields and status.
    """
    if pw_row is None:
        return {
            "has_pw_region": False,
            "metadata_qc": "fail",
            "metadata_qc_reason": "no_pw_spectral_region_found",
            "display_pages_per_pixel": None,
            "velocity_formula": "",
        }

    reasons = []
    x0 = pw_row.get("X0")
    x1 = pw_row.get("X1")
    y0 = pw_row.get("Y0")
    y1 = pw_row.get("Y1")
    vir_y = pw_row.get("VirY")
    baseline_global = pw_row.get("baseline_y_global_px")
    phy_delta_x = pw_row.get("PhyDeltaX")
    phy_delta_y = pw_row.get("PhyDeltaY")
    roi_width = pw_row.get("roi_width_px")
    roi_height = pw_row.get("roi_height_px")

    if roi_width is None or roi_width <= 0:
        reasons.append("invalid_roi_width")
    if roi_height is None or roi_height <= 0:
        reasons.append("invalid_roi_height")
    if vir_y is None:
        reasons.append("missing_VirY")
    elif roi_height is not None and not (0 <= vir_y <= roi_height):
        reasons.append("VirY_outside_roi")
    if baseline_global is None:
        reasons.append("missing_baseline_global")
    elif y0 is not None and y1 is not None and not (y0 <= baseline_global <= y1):
        reasons.append("baseline_outside_roi")
    if phy_delta_x is None:
        reasons.append("missing_PhyDeltaX")
    elif not (0.001 <= abs(phy_delta_x) <= 0.02):
        reasons.append("unexpected_PhyDeltaX")
    if phy_delta_y is None:
        reasons.append("missing_PhyDeltaY")
    elif not (0.01 <= abs(phy_delta_y) <= 1.0):
        reasons.append("unexpected_PhyDeltaY")

    display_pages_per_pixel = None

    if phy_delta_x is not None:
        display_pages_per_pixel = PAGE_RATE_HZ * phy_delta_x

        if not (2.0 <= abs(display_pages_per_pixel) <= 4.0):
            reasons.append("unexpected_pages_per_pixel")

    if len(reasons) == 0:
        metadata_qc = "pass"
        metadata_qc_reason = "pw_region_and_calibration_plausible"
    elif any(reason.startswith("missing") or reason.startswith("invalid") for reason in reasons):
        metadata_qc = "fail"
        metadata_qc_reason = ";".join(reasons)
    else:
        metadata_qc = "warning"
        metadata_qc_reason = ";".join(reasons)

    velocity_formula = ""
    if baseline_global is not None and phy_delta_y is not None:
        velocity_formula = (f"v_cm_s = (y_full_px - {baseline_global}) * ({phy_delta_y})")

    return {
        "has_pw_region": True,
        "metadata_qc": metadata_qc,
        "metadata_qc_reason": metadata_qc_reason,
        "display_pages_per_pixel": round(float(display_pages_per_pixel), 6)
        if display_pages_per_pixel is not None else None,
        "velocity_formula": velocity_formula,
    }


def build_pw_calibration_qc_table(native_inventory_df):
    """
    Build PW calibration QC table for all native recordings.

    Parameters
    ----------
    native_inventory_df : pandas.DataFrame
        Native inventory table from Cell group 2.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        - pw_calibration_qc_df: one row per recording
        - dcm_region_summary_df: one row per DcmRegionPara region
    """
    calibration_rows = []
    all_region_rows = []

    for _, inv_row in native_inventory_df.iterrows():
        recording_id = inv_row["recording_id"]
        dcm_path = inv_row.get("dcm_region_para_path", "")
        parsed = parse_dcm_region_para_file(dcm_path)
        region_rows = build_dcm_region_summary_rows(recording_id=recording_id, parsed_dcm=parsed,)
        all_region_rows.extend(region_rows)
        pw_row = select_pw_spectral_region(region_rows)
        qc = evaluate_pw_calibration_qc(pw_row)

        base_row = {
            "recording_id": recording_id,
            "dcm_parse_ok": parsed["parse_ok"],
            "dcm_parse_error": parsed["error"],
            "dcm_n_regions_declared": parsed["n_regions_declared"],
            "dcm_n_regions_parsed": len(parsed["regions"]),

            "pw_region_name": pw_row.get("region_name") if pw_row else "",
            "pw_DataType": pw_row.get("DataType") if pw_row else "",
            "pw_SpatialFormat": pw_row.get("SpatialFormat") if pw_row else "",
            "pw_PhyUnitsX": pw_row.get("PhyUnitsX") if pw_row else "",
            "pw_PhyUnitsY": pw_row.get("PhyUnitsY") if pw_row else "",

            "pw_roi_x0": pw_row.get("X0") if pw_row else None,
            "pw_roi_x1": pw_row.get("X1") if pw_row else None,
            "pw_roi_y0": pw_row.get("Y0") if pw_row else None,
            "pw_roi_y1": pw_row.get("Y1") if pw_row else None,
            "pw_roi_width_px": pw_row.get("roi_width_px") if pw_row else None,
            "pw_roi_height_px": pw_row.get("roi_height_px") if pw_row else None,

            "pw_baseline_y_roi_px": pw_row.get("VirY") if pw_row else None,
            "pw_baseline_y_global_px": pw_row.get("baseline_y_global_px") if pw_row else None,

            "pw_phy_delta_x_s_per_px": pw_row.get("PhyDeltaX") if pw_row else None,
            "pw_phy_delta_y_cm_s_per_px": pw_row.get("PhyDeltaY") if pw_row else None,
        }

        base_row.update(qc)
        calibration_rows.append(base_row)

    pw_calibration_qc_df = pd.DataFrame(calibration_rows)
    dcm_region_summary_df = pd.DataFrame(all_region_rows)

    if len(pw_calibration_qc_df) > 0:
        pw_calibration_qc_df = pw_calibration_qc_df.sort_values("recording_id").reset_index(drop=True)
    if len(dcm_region_summary_df) > 0:
        dcm_region_summary_df = dcm_region_summary_df.sort_values(["recording_id", "region_name"]).reset_index(drop=True)

    return pw_calibration_qc_df, dcm_region_summary_df


def summarize_pw_calibration_qc(pw_calibration_qc_df):
    """
    Print compact summary of PW calibration QC.

    Parameters
    ----------
    pw_calibration_qc_df : pandas.DataFrame
        PW calibration QC table.
    """

    print("PW calibration QC")
    print("=" * 80)

    if len(pw_calibration_qc_df) == 0:
        print("No rows.")
        return

    print(f"Rows: {len(pw_calibration_qc_df)}")
    print()

    print("metadata_qc counts:")
    print(pw_calibration_qc_df["metadata_qc"].value_counts(dropna=False))
    print()

    summary_cols = [
        "recording_id",
        "metadata_qc",
        "pw_region_name",
        "pw_roi_x0",
        "pw_roi_x1",
        "pw_roi_y0",
        "pw_roi_y1",
        "pw_baseline_y_global_px",
        "pw_phy_delta_x_s_per_px",
        "pw_phy_delta_y_cm_s_per_px",
        "display_pages_per_pixel",
    ]

    display_cols = [col for col in summary_cols if col in pw_calibration_qc_df.columns]
    display(pw_calibration_qc_df[display_cols])

    print()
    print("Unique calibration values:")
    unique_cols = [
        "pw_roi_x0",
        "pw_roi_x1",
        "pw_roi_y0",
        "pw_roi_y1",
        "pw_baseline_y_global_px",
        "pw_phy_delta_x_s_per_px",
        "pw_phy_delta_y_cm_s_per_px",
        "display_pages_per_pixel",
    ]

    for col in unique_cols:
        if col in pw_calibration_qc_df.columns:
            values = sorted(pw_calibration_qc_df[col].dropna().unique().tolist())
            print(f"{col}: {values}")


def save_pw_calibration_qc_tables(pw_calibration_qc_df, dcm_region_summary_df, output_dir,):
    """
    Save PW calibration QC and DcmRegion summary tables.

    Parameters
    ----------
    pw_calibration_qc_df : pandas.DataFrame
        One row per recording.

    dcm_region_summary_df : pandas.DataFrame
        One row per DcmRegionPara region.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    dict
        Saved output paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    calibration_path = output_dir / "nb06_v2_pw_calibration_qc.csv"
    region_summary_path = output_dir / "nb06_v2_dcm_region_summary.csv"

    pw_calibration_qc_df.to_csv(calibration_path, index=False)
    dcm_region_summary_df.to_csv(region_summary_path, index=False)

    print(f"Saved PW calibration QC CSV: {calibration_path}")
    print(f"Saved DcmRegion summary CSV: {region_summary_path}")

    return {
        "pw_calibration_qc_csv": calibration_path,
        "dcm_region_summary_csv": region_summary_path,
    }



In [7]:
pw_calibration_qc_df, dcm_region_summary_df = build_pw_calibration_qc_table(native_inventory_df)
summarize_pw_calibration_qc(pw_calibration_qc_df)

pw_calibration_output_paths = save_pw_calibration_qc_tables(pw_calibration_qc_df, dcm_region_summary_df, PATHS["NB06_V2_REPORTS_DIR"])
pw_calibration_qc_df

PW calibration QC
Rows: 10

metadata_qc counts:
metadata_qc
pass    10
Name: count, dtype: int64



,recording_id,metadata_qc,pw_region_name,pw_roi_x0,pw_roi_x1,pw_roi_y0,pw_roi_y1,pw_baseline_y_global_px,pw_phy_delta_x_s_per_px,pw_phy_delta_y_cm_s_per_px,display_pages_per_pixel
0,202606130411060002SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
1,202606130413540003SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
2,202606130417060004SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
3,202606130420260005SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
4,202606130422440006SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
5,202606130426500007SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
6,202606130430380008SMP,pass,DcmRegion0,79,659,245,508,376,0.006,-0.176946,3.0
7,202606130433280009SMP,pass,DcmRegion0,79,659,245,508,376,0.006,-0.176946,3.0
8,202606130434130010SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0
9,202606130437480012SMP,pass,DcmRegion1,79,659,245,508,376,0.006,-0.176946,3.0



Unique calibration values:
pw_roi_x0: [79]
pw_roi_x1: [659]
pw_roi_y0: [245]
pw_roi_y1: [508]
pw_baseline_y_global_px: [376]
pw_phy_delta_x_s_per_px: [0.0060000000521541]
pw_phy_delta_y_cm_s_per_px: [-0.1769457646367944]
display_pages_per_pixel: [3.0]
Saved PW calibration QC CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_pw_calibration_qc.csv
Saved DcmRegion summary CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_dcm_region_summary.csv


,recording_id,dcm_parse_ok,dcm_parse_error,dcm_n_regions_declared,dcm_n_regions_parsed,pw_region_name,pw_DataType,pw_SpatialFormat,pw_PhyUnitsX,pw_PhyUnitsY,...,pw_roi_height_px,pw_baseline_y_roi_px,pw_baseline_y_global_px,pw_phy_delta_x_s_per_px,pw_phy_delta_y_cm_s_per_px,has_pw_region,metadata_qc,metadata_qc_reason,display_pages_per_pixel,velocity_formula
0,202606130411060002SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
1,202606130413540003SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
2,202606130417060004SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
3,202606130420260005SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
4,202606130422440006SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
5,202606130426500007SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
6,202606130430380008SMP,True,,2,2,DcmRegion0,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
7,202606130433280009SMP,True,,2,2,DcmRegion0,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
8,202606130434130010SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...
9,202606130437480012SMP,True,,2,2,DcmRegion1,3,3,4,7,...,263,131,376,0.006,-0.176946,True,pass,pw_region_and_calibration_plausible,3.0,v_cm_s = (y_full_px - 376) * (-0.1769457646367...


## Native / AVI / audio duration agreement

This section compares recording duration from independent sources:

- native PW binary pages: `n_pages / 500`,
- linked AVI video stream,
- linked AVI audio stream,
- optional `app.xml` declared duration.

The native page-count duration is the primary native timing reference.

Important:

- `app.xml` duration may be integer-rounded, so it is treated as secondary.
- Timing QC is based mainly on pages vs AVI/audio agreement.
- This is duration/timing consistency only, not beat timing.

In [8]:
def parse_fraction_to_float(value):
    """
    Parse a fraction-like string into float.

    Examples
    --------
    "30/1" -> 30.0
    "30000/1001" -> 29.97
    "30" -> 30.0

    Parameters
    ----------
    value : str or numeric
        Fraction-like value.

    Returns
    -------
    float or None
        Parsed float value.
    """
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)

    value = str(value).strip()

    if value == "":
        return None

    if "/" in value:
        try:
            numerator, denominator = value.split("/", 1)
            numerator = float(numerator)
            denominator = float(denominator)

            if denominator == 0:
                return None

            return numerator / denominator

        except Exception:
            return None

    try:
        return float(value)
    except Exception:
        return None


def run_ffprobe_json(media_path):
    """
    Run ffprobe and return parsed JSON metadata.

    Parameters
    ----------
    media_path : pathlib.Path or str
        Path to media file.

    Returns
    -------
    dict or None
        Parsed ffprobe JSON, or None if ffprobe is unavailable/fails.
    """
    media_path = Path(media_path)

    if not media_path.exists():
        return None

    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_streams",
        "-show_format",
        "-of",
        "json",
        str(media_path),
    ]

    try:
        completed = subprocess.run(cmd, capture_output=True, text=True, check=False)
    except FileNotFoundError:
        return None
    except Exception:
        return None

    if completed.returncode != 0:
        return None

    try:
        return json.loads(completed.stdout)
    except Exception:
        return None


def get_media_durations_with_ffprobe(media_path):
    """
    Read video/audio/format duration from ffprobe.

    Parameters
    ----------
    media_path : pathlib.Path or str
        Path to AVI/MP4.

    Returns
    -------
    dict
        Media duration metadata.
    """
    result = {
        "ffprobe_ok": False,
        "video_duration_s": None,
        "audio_duration_s": None,
        "format_duration_s": None,
        "video_frame_count": None,
        "video_fps": None,
        "duration_source": "unavailable",
        "error": "",
    }

    media_path = Path(media_path)
    probe = run_ffprobe_json(media_path)

    if probe is None:
        result["error"] = "ffprobe_unavailable_or_failed"
        return result

    result["ffprobe_ok"] = True
    format_info = probe.get("format", {})
    format_duration = format_info.get("duration")

    try:
        result["format_duration_s"] = float(format_duration)
    except Exception:
        result["format_duration_s"] = None

    streams = probe.get("streams", [])
    video_streams = [stream for stream in streams if stream.get("codec_type") == "video"]
    audio_streams = [stream for stream in streams if stream.get("codec_type") == "audio"]

    if len(video_streams) > 0:
        video = video_streams[0]
        fps = parse_fraction_to_float(video.get("avg_frame_rate") or video.get("r_frame_rate"))
        result["video_fps"] = fps
        nb_frames = video.get("nb_frames")

        try:
            frame_count = int(nb_frames)
        except Exception:
            frame_count = None

        result["video_frame_count"] = frame_count
        stream_duration = video.get("duration")
        video_duration = None

        if frame_count is not None and fps is not None and fps > 0:
            video_duration = frame_count / fps
            result["duration_source"] = "ffprobe_nb_frames_over_fps"

        elif stream_duration is not None:
            try:
                video_duration = float(stream_duration)
                result["duration_source"] = "ffprobe_video_stream_duration"
            except Exception:
                video_duration = None

        elif result["format_duration_s"] is not None:
            video_duration = result["format_duration_s"]
            result["duration_source"] = "ffprobe_format_duration"

        result["video_duration_s"] = video_duration

    if len(audio_streams) > 0:
        audio = audio_streams[0]
        audio_duration = None

        if audio.get("duration") is not None:
            try:
                audio_duration = float(audio.get("duration"))
            except Exception:
                audio_duration = None

        if audio_duration is None and result["format_duration_s"] is not None:
            audio_duration = result["format_duration_s"]

        result["audio_duration_s"] = audio_duration

    return result


def get_video_duration_with_cv2(media_path):
    """
    Fallback video duration reader using OpenCV.

    Parameters
    ----------
    media_path : pathlib.Path or str
        Path to AVI/MP4.

    Returns
    -------
    dict
        Video duration metadata.
    """
    result = {
        "cv2_ok": False,
        "video_duration_s": None,
        "video_frame_count": None,
        "video_fps": None,
        "error": "",
    }

    try:
        import cv2
    except Exception:
        result["error"] = "opencv_unavailable"
        return result

    media_path = Path(media_path)

    if not media_path.exists():
        result["error"] = "file_not_found"
        return result

    cap = cv2.VideoCapture(str(media_path))

    if not cap.isOpened():
        result["error"] = "cv2_open_failed"
        return result

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))

    cap.release()

    if frame_count <= 0 or fps <= 0:
        result["error"] = "invalid_frame_count_or_fps"
        return result

    result["cv2_ok"] = True
    result["video_frame_count"] = frame_count
    result["video_fps"] = fps
    result["video_duration_s"] = frame_count / fps

    return result


def get_linked_avi_duration_metadata(media_path):
    """
    Get linked AVI duration metadata using ffprobe first, OpenCV fallback second.

    Parameters
    ----------
    media_path : pathlib.Path or str
        Path to linked AVI.

    Returns
    -------
    dict
        Duration metadata.
    """
    media_path = Path(media_path)

    result = {
        "media_path": str(media_path),
        "media_exists": media_path.exists(),
        "video_duration_s": None,
        "audio_duration_s": None,
        "format_duration_s": None,
        "video_frame_count": None,
        "video_fps": None,
        "media_duration_source": "unavailable",
        "ffprobe_ok": False,
        "cv2_ok": False,
        "media_error": "",
    }

    if not media_path.exists():
        result["media_error"] = "file_not_found"
        return result

    ffprobe_result = get_media_durations_with_ffprobe(media_path)
    result["ffprobe_ok"] = ffprobe_result["ffprobe_ok"]

    if ffprobe_result["ffprobe_ok"]:
        result["video_duration_s"] = ffprobe_result["video_duration_s"]
        result["audio_duration_s"] = ffprobe_result["audio_duration_s"]
        result["format_duration_s"] = ffprobe_result["format_duration_s"]
        result["video_frame_count"] = ffprobe_result["video_frame_count"]
        result["video_fps"] = ffprobe_result["video_fps"]
        result["media_duration_source"] = ffprobe_result["duration_source"]
        return result

    cv2_result = get_video_duration_with_cv2(media_path)
    result["cv2_ok"] = cv2_result["cv2_ok"]

    if cv2_result["cv2_ok"]:
        result["video_duration_s"] = cv2_result["video_duration_s"]
        result["video_frame_count"] = cv2_result["video_frame_count"]
        result["video_fps"] = cv2_result["video_fps"]
        result["media_duration_source"] = "cv2_frame_count_over_fps"
        result["media_error"] = "ffprobe_failed_used_cv2_no_audio_duration"
        return result

    result["media_error"] = (f"ffprobe_failed={ffprobe_result.get('error', '')}; cv2_failed={cv2_result.get('error', '')}")

    return result


def parse_app_xml_duration_seconds(app_xml_path):
    """
    Parse app.xml for declared recording duration.

    This is secondary metadata. In the Mindray exports seen so far, app.xml
    may provide integer-second duration or other rounded timing fields.

    Parameters
    ----------
    app_xml_path : pathlib.Path or str
        Path to app.xml.

    Returns
    -------
    dict
        Parsed app.xml timing metadata.
    """
    result = {
        "app_xml_exists": False,
        "app_xml_duration_s": None,
        "app_xml_duration_source": "",
        "app_xml_error": "",
    }

    app_xml_path = Path(app_xml_path)

    if not app_xml_path.exists():
        result["app_xml_error"] = "file_not_found"
        return result

    result["app_xml_exists"] = True

    try:
        text = app_xml_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    except Exception as exc:
        result["app_xml_error"] = f"read_error: {exc}"
        return result

    patterns = [
        ("Duration_tag", r"<Duration>\s*([0-9]+(?:\.[0-9]+)?)\s*</Duration>"),
        ("Duration_attribute", r"Duration\s*=\s*['\"]?([0-9]+(?:\.[0-9]+)?)"),
        ("ImageTime_tag", r"<ImageTime>\s*([0-9]+(?:\.[0-9]+)?)\s*</ImageTime>"),
        ("CaptureTime_tag", r"<CaptureTime>\s*([0-9]+(?:\.[0-9]+)?)\s*</CaptureTime>"),
    ]

    for source_name, pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)

        if match is None:
            continue

        try:
            result["app_xml_duration_s"] = float(match.group(1))
            result["app_xml_duration_source"] = source_name
            return result
        except Exception:
            continue

    result["app_xml_error"] = "duration_not_found"

    return result


def evaluate_duration_agreement(
    duration_pages_s,
    duration_avi_s,
    duration_audio_s,
    duration_app_xml_s,
    primary_pass_tolerance_s=0.20,
    primary_caution_tolerance_s=0.50,
    app_xml_ok_tolerance_s=1.20,
):
    """
    Evaluate duration agreement across native pages, AVI video, AVI audio, and app.xml.

    Timing QC is based on primary media sources:
    - pages vs AVI
    - pages vs audio

    app.xml is secondary because it may be integer-rounded.

    Parameters
    ----------
    duration_pages_s : float
        Native duration from n_pages / PAGE_RATE_HZ.

    duration_avi_s : float or None
        Linked AVI video duration.

    duration_audio_s : float or None
        Linked AVI audio duration.

    duration_app_xml_s : float or None
        app.xml declared duration.

    primary_pass_tolerance_s : float
        Max discrepancy for pass.

    primary_caution_tolerance_s : float
        Max discrepancy for caution.

    app_xml_ok_tolerance_s : float
        Tolerance for secondary app.xml agreement.

    Returns
    -------
    dict
        Duration QC fields.
    """
    discrepancies = {}

    if duration_pages_s is not None and duration_avi_s is not None:
        discrepancies["pages_vs_avi_s"] = abs(duration_pages_s - duration_avi_s)
    else:
        discrepancies["pages_vs_avi_s"] = None

    if duration_pages_s is not None and duration_audio_s is not None:
        discrepancies["pages_vs_audio_s"] = abs(duration_pages_s - duration_audio_s)
    else:
        discrepancies["pages_vs_audio_s"] = None

    if duration_pages_s is not None and duration_app_xml_s is not None:
        discrepancies["pages_vs_app_xml_s"] = abs(duration_pages_s - duration_app_xml_s)
    else:
        discrepancies["pages_vs_app_xml_s"] = None

    primary_values = [
        value for key, value in discrepancies.items()
        if key in ("pages_vs_avi_s", "pages_vs_audio_s")
        and value is not None
    ]

    if len(primary_values) == 0:
        timing_qc = "unavailable"
        timing_qc_reason = "no_primary_media_duration_available"
        max_primary_discrepancy_s = None

    else:
        max_primary_discrepancy_s = max(primary_values)

        if max_primary_discrepancy_s <= primary_pass_tolerance_s:
            timing_qc = "pass"
            timing_qc_reason = (
                f"max_primary_discrepancy_s<={primary_pass_tolerance_s}"
            )

        elif max_primary_discrepancy_s <= primary_caution_tolerance_s:
            timing_qc = "caution"
            timing_qc_reason = (
                f"max_primary_discrepancy_s<={primary_caution_tolerance_s}"
            )

        else:
            timing_qc = "fail"
            timing_qc_reason = (
                f"max_primary_discrepancy_s>{primary_caution_tolerance_s}"
            )

    app_xml_discrepancy = discrepancies["pages_vs_app_xml_s"]

    if app_xml_discrepancy is None:
        app_xml_qc = "unavailable"
    elif app_xml_discrepancy <= app_xml_ok_tolerance_s:
        app_xml_qc = "ok_integer_or_rounded"
    else:
        app_xml_qc = "warning"

    return {
        "pages_vs_avi_s": round(discrepancies["pages_vs_avi_s"], 4)
        if discrepancies["pages_vs_avi_s"] is not None else None,
        "pages_vs_audio_s": round(discrepancies["pages_vs_audio_s"], 4)
        if discrepancies["pages_vs_audio_s"] is not None else None,
        "pages_vs_app_xml_s": round(discrepancies["pages_vs_app_xml_s"], 4)
        if discrepancies["pages_vs_app_xml_s"] is not None else None,
        "max_primary_discrepancy_s": round(max_primary_discrepancy_s, 4)
        if max_primary_discrepancy_s is not None else None,
        "timing_qc": timing_qc,
        "timing_qc_reason": timing_qc_reason,
        "app_xml_qc": app_xml_qc,
    }


def build_duration_qc_table(native_inventory_df):
    """
    Build duration agreement table for all native recordings.

    Parameters
    ----------
    native_inventory_df : pandas.DataFrame
        Native inventory table.

    Returns
    -------
    pandas.DataFrame
        One row per recording with duration agreement metadata.
    """
    rows = []

    for _, inv_row in native_inventory_df.iterrows():
        recording_id = inv_row["recording_id"]

        duration_pages_s = inv_row.get("duration_pages_s")
        linked_avi_path = inv_row.get("linked_avi_path", "")
        app_xml_path = inv_row.get("app_xml_path", "")

        media_meta = get_linked_avi_duration_metadata(linked_avi_path)
        app_meta = parse_app_xml_duration_seconds(app_xml_path)

        duration_avi_s = media_meta["video_duration_s"]
        duration_audio_s = media_meta["audio_duration_s"]
        duration_app_xml_s = app_meta["app_xml_duration_s"]

        qc = evaluate_duration_agreement(
            duration_pages_s=duration_pages_s,
            duration_avi_s=duration_avi_s,
            duration_audio_s=duration_audio_s,
            duration_app_xml_s=duration_app_xml_s,
        )

        row = {
            "recording_id": recording_id,

            "duration_pages_s": round(float(duration_pages_s), 4)
            if duration_pages_s is not None else None,

            "duration_avi_s": round(float(duration_avi_s), 4)
            if duration_avi_s is not None else None,

            "duration_audio_s": round(float(duration_audio_s), 4)
            if duration_audio_s is not None else None,

            "duration_format_s": round(float(media_meta["format_duration_s"]), 4)
            if media_meta["format_duration_s"] is not None else None,

            "duration_app_xml_s": round(float(duration_app_xml_s), 4)
            if duration_app_xml_s is not None else None,

            "video_frame_count": media_meta["video_frame_count"],
            "video_fps": round(float(media_meta["video_fps"]), 6)
            if media_meta["video_fps"] is not None else None,

            "media_duration_source": media_meta["media_duration_source"],
            "ffprobe_ok": media_meta["ffprobe_ok"],
            "cv2_ok": media_meta["cv2_ok"],
            "media_error": media_meta["media_error"],

            "app_xml_exists": app_meta["app_xml_exists"],
            "app_xml_duration_source": app_meta["app_xml_duration_source"],
            "app_xml_error": app_meta["app_xml_error"],
        }

        row.update(qc)
        rows.append(row)

    duration_qc_df = pd.DataFrame(rows)

    if len(duration_qc_df) > 0:
        duration_qc_df = duration_qc_df.sort_values(
            "recording_id"
        ).reset_index(drop=True)

    return duration_qc_df


def summarize_duration_qc(duration_qc_df):
    """
    Print compact summary of duration QC.

    Parameters
    ----------
    duration_qc_df : pandas.DataFrame
        Duration QC table.
    """

    print("Duration agreement QC")
    print("=" * 80)

    if len(duration_qc_df) == 0:
        print("No rows.")
        return

    print(f"Rows: {len(duration_qc_df)}")
    print()

    print("timing_qc counts:")
    print(duration_qc_df["timing_qc"].value_counts(dropna=False))
    print()

    print("app_xml_qc counts:")
    print(duration_qc_df["app_xml_qc"].value_counts(dropna=False))
    print()

    summary_cols = [
        "recording_id",
        "duration_pages_s",
        "duration_avi_s",
        "duration_audio_s",
        "duration_app_xml_s",
        "pages_vs_avi_s",
        "pages_vs_audio_s",
        "pages_vs_app_xml_s",
        "max_primary_discrepancy_s",
        "timing_qc",
        "app_xml_qc",
        "video_frame_count",
        "video_fps",
        "media_duration_source",
    ]

    summary_cols = [col for col in summary_cols if col in duration_qc_df.columns]
    display(duration_qc_df[summary_cols])

    print()
    print("Max discrepancies:")
    for col in ["pages_vs_avi_s", "pages_vs_audio_s", "pages_vs_app_xml_s"]:
        if col in duration_qc_df.columns:
            max_val = duration_qc_df[col].dropna().max()
            print(f"{col}: {max_val}")


def save_duration_qc_table(duration_qc_df, output_dir):
    """
    Save duration QC table.

    Parameters
    ----------
    duration_qc_df : pandas.DataFrame
        Duration QC table.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    pathlib.Path
        Saved CSV path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "nb06_v2_duration_qc.csv"

    duration_qc_df.to_csv(output_path, index=False)

    print(f"Saved duration QC CSV: {output_path}")

    return output_path


In [9]:
duration_qc_df = build_duration_qc_table(native_inventory_df)
summarize_duration_qc(duration_qc_df)
duration_qc_csv_path = save_duration_qc_table(duration_qc_df, PATHS["NB06_V2_REPORTS_DIR"],)
duration_qc_df

Duration agreement QC
Rows: 10

timing_qc counts:
timing_qc
pass    10
Name: count, dtype: int64

app_xml_qc counts:
app_xml_qc
unavailable    10
Name: count, dtype: int64



,recording_id,duration_pages_s,duration_avi_s,duration_audio_s,duration_app_xml_s,pages_vs_avi_s,pages_vs_audio_s,pages_vs_app_xml_s,max_primary_discrepancy_s,timing_qc,app_xml_qc,video_frame_count,video_fps,media_duration_source
0,202606130411060002SMP,59.230,59.2667,59.2427,None,0.0367,0.0127,None,0.0367,pass,unavailable,1778,30.0,ffprobe_nb_frames_over_fps
1,202606130413540003SMP,46.982,47.0333,46.9973,None,0.0513,0.0153,None,0.0513,pass,unavailable,1411,30.0,ffprobe_nb_frames_over_fps
2,202606130417060004SMP,42.464,42.5000,42.4747,None,0.0360,0.0107,None,0.0360,pass,unavailable,1275,30.0,ffprobe_nb_frames_over_fps
3,202606130420260005SMP,45.528,45.5667,45.5467,None,0.0387,0.0187,None,0.0387,pass,unavailable,1367,30.0,ffprobe_nb_frames_over_fps
4,202606130422440006SMP,51.252,51.3000,51.2640,None,0.0480,0.0120,None,0.0480,pass,unavailable,1539,30.0,ffprobe_nb_frames_over_fps
5,202606130426500007SMP,36.692,36.7333,36.7040,None,0.0413,0.0120,None,0.0413,pass,unavailable,1102,30.0,ffprobe_nb_frames_over_fps
6,202606130430380008SMP,44.372,44.4333,44.3840,None,0.0613,0.0120,None,0.0613,pass,unavailable,1333,30.0,ffprobe_nb_frames_over_fps
7,202606130433280009SMP,31.724,31.7667,31.7333,None,0.0427,0.0093,None,0.0427,pass,unavailable,953,30.0,ffprobe_nb_frames_over_fps
8,202606130434130010SMP,33.832,33.8667,33.8453,None,0.0347,0.0133,None,0.0347,pass,unavailable,1016,30.0,ffprobe_nb_frames_over_fps
9,202606130437480012SMP,45.026,45.0667,45.0453,None,0.0407,0.0193,None,0.0407,pass,unavailable,1352,30.0,ffprobe_nb_frames_over_fps



Max discrepancies:
pages_vs_avi_s: 0.0613
pages_vs_audio_s: 0.0193
pages_vs_app_xml_s: nan
Saved duration QC CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_duration_qc.csv


,recording_id,duration_pages_s,duration_avi_s,duration_audio_s,duration_format_s,duration_app_xml_s,video_frame_count,video_fps,media_duration_source,ffprobe_ok,...,app_xml_exists,app_xml_duration_source,app_xml_error,pages_vs_avi_s,pages_vs_audio_s,pages_vs_app_xml_s,max_primary_discrepancy_s,timing_qc,timing_qc_reason,app_xml_qc
0,202606130411060002SMP,59.230,59.2667,59.2427,59.2667,None,1778,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0367,0.0127,None,0.0367,pass,max_primary_discrepancy_s<=0.2,unavailable
1,202606130413540003SMP,46.982,47.0333,46.9973,47.0333,None,1411,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0513,0.0153,None,0.0513,pass,max_primary_discrepancy_s<=0.2,unavailable
2,202606130417060004SMP,42.464,42.5000,42.4747,42.5000,None,1275,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0360,0.0107,None,0.0360,pass,max_primary_discrepancy_s<=0.2,unavailable
3,202606130420260005SMP,45.528,45.5667,45.5467,45.5667,None,1367,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0387,0.0187,None,0.0387,pass,max_primary_discrepancy_s<=0.2,unavailable
4,202606130422440006SMP,51.252,51.3000,51.2640,51.3000,None,1539,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0480,0.0120,None,0.0480,pass,max_primary_discrepancy_s<=0.2,unavailable
5,202606130426500007SMP,36.692,36.7333,36.7040,36.7333,None,1102,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0413,0.0120,None,0.0413,pass,max_primary_discrepancy_s<=0.2,unavailable
6,202606130430380008SMP,44.372,44.4333,44.3840,44.4333,None,1333,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0613,0.0120,None,0.0613,pass,max_primary_discrepancy_s<=0.2,unavailable
7,202606130433280009SMP,31.724,31.7667,31.7333,31.7667,None,953,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0427,0.0093,None,0.0427,pass,max_primary_discrepancy_s<=0.2,unavailable
8,202606130434130010SMP,33.832,33.8667,33.8453,33.8667,None,1016,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0347,0.0133,None,0.0347,pass,max_primary_discrepancy_s<=0.2,unavailable
9,202606130437480012SMP,45.026,45.0667,45.0453,45.0667,None,1352,30.0,ffprobe_nb_frames_over_fps,True,...,True,,duration_not_found,0.0407,0.0193,None,0.0407,pass,max_primary_discrepancy_s<=0.2,unavailable


## Manual native-to-NB04 mapping and NB04 availability

This section attaches each native recording to its confirmed NB04 V2 recording name.

Important:

- Mapping is not inferred blindly from duration.
- Ambiguous/conflicting recordings are explicitly marked.
- Native recordings with no valid NB04 counterpart remain useful for native-only QC/documentation.
- NB04 audio and image peak availability are checked from NB04 V2 exports.

This section does not perform native beat timing.

In [10]:
# Manual mapping validated during native-to-NB04 ambiguity review.
# This is used as a safe fallback and as a clear project-level decision log.
MANUAL_NATIVE_TO_NB04_DECISIONS = {
    "202606130411060002SMP": {
        "nb04_recording_name": "candidate_test_02_brachial",
        "nb04_match_status": "confirmed",
        "reason": "manual_validated_mapping",
    },
    "202606130413540003SMP": {
        "nb04_recording_name": "candidate_test_03_brachial",
        "nb04_match_status": "confirmed",
        "reason": "manual_validated_mapping",
    },
    "202606130417060004SMP": {
        "nb04_recording_name": "",
        "nb04_match_status": "no_match",
        "reason": "no_valid_nb04_counterpart",
    },
    "202606130420260005SMP": {
        "nb04_recording_name": "candidate_test_08_brachial",
        "nb04_match_status": "confirmed",
        "reason": "confirmed_by_frame_count",
    },
    "202606130422440006SMP": {
        "nb04_recording_name": "candidate_test_02_neck",
        "nb04_match_status": "confirmed",
        "reason": "manual_validated_mapping",
    },
    "202606130426500007SMP": {
        "nb04_recording_name": "candidate_test_05_brachial",
        "nb04_match_status": "confirmed",
        "reason": "manual_validated_mapping",
    },
    "202606130430380008SMP": {
        "nb04_recording_name": "",
        "nb04_match_status": "no_match",
        "reason": "duration_conflict_no_valid_nb04_counterpart",
    },
    "202606130433280009SMP": {
        "nb04_recording_name": "candidate_test_04_brachial_2",
        "nb04_match_status": "confirmed",
        "reason": "manual_validated_mapping",
    },
    "202606130434130010SMP": {
        "nb04_recording_name": "candidate_test_04_brachial_1",
        "nb04_match_status": "confirmed",
        "reason": "manual_validated_mapping",
    },
    "202606130437480012SMP": {
        "nb04_recording_name": "candidate_test_01_brachial",
        "nb04_match_status": "confirmed",
        "reason": "confirmed_by_frame_count",
    },
}

In [14]:
def load_optional_csv(csv_path):
    """
    Load a CSV file if it exists.

    Parameters
    ----------
    csv_path : pathlib.Path or str
        CSV path.

    Returns
    -------
    pandas.DataFrame
        Loaded table, or empty DataFrame if missing.
    """
    csv_path = Path(csv_path)

    if not csv_path.exists():
        warnings.warn(f"CSV not found: {csv_path}")
        return pd.DataFrame()

    return pd.read_csv(csv_path)


def first_existing_column(df, candidate_columns):
    """
    Return the first column name that exists in a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame.

    candidate_columns : list[str]
        Candidate column names.

    Returns
    -------
    str or None
        First existing column, or None.
    """
    for col in candidate_columns:
        if col in df.columns:
            return col

    return None


def clean_string_value(value):
    """
    Convert a value to a stripped string, returning empty string for NaN/None.

    Parameters
    ----------
    value : object
        Input value.

    Returns
    -------
    str
        Clean string.
    """
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def build_manual_mapping_fallback_df(native_inventory_df):
    """
    Build native-to-NB04 mapping table from validated project decisions.

    Parameters
    ----------
    native_inventory_df : pandas.DataFrame
        Native inventory table.

    Returns
    -------
    pandas.DataFrame
        Mapping table.
    """
    rows = []

    for recording_id in native_inventory_df["recording_id"].tolist():
        decision = MANUAL_NATIVE_TO_NB04_DECISIONS.get(recording_id)

        if decision is None:
            rows.append(
                {
                    "recording_id": recording_id,
                    "nb04_recording_name": "",
                    "nb04_match_status": "unreviewed",
                    "nb04_mapping_source": "manual_fallback_missing",
                    "nb04_mapping_reason": "recording_not_in_manual_decision_dict",
                }
            )
            continue

        rows.append(
            {
                "recording_id": recording_id,
                "nb04_recording_name": decision["nb04_recording_name"],
                "nb04_match_status": decision["nb04_match_status"],
                "nb04_mapping_source": "manual_validated_decision_dict",
                "nb04_mapping_reason": decision["reason"],
            }
        )

    return pd.DataFrame(rows)


def build_mapping_from_template_csv(template_csv_path, native_inventory_df):
    """
    Build native-to-NB04 mapping table.

    NB06 V2 priority rule:

    1. MANUAL_NATIVE_TO_NB04_DECISIONS is the source of truth.
       These are the validated final project decisions after ambiguity review.

    2. native_to_nb04_manual_match_template.csv is used only as a diagnostic
       sidecar, because it may still contain old AMBIGUOUS / CONFLICT suggestions.

    3. Do not downgrade a validated manual decision because the template still
       says AMBIGUOUS or CONFLICT.

    Parameters
    ----------
    template_csv_path : pathlib.Path or str
        Path to native_to_nb04_manual_match_template.csv.

    native_inventory_df : pandas.DataFrame
        Native inventory table.

    Returns
    -------
    pandas.DataFrame
        Mapping table with final NB06 V2 mapping decisions.
    """

    template_csv_path = Path(template_csv_path)

    template_by_id = {}

    if template_csv_path.exists():
        template_df = pd.read_csv(template_csv_path)

        native_id_col = first_existing_column(
            template_df,
            [
                "native_recording_id",
                "recording_id",
                "native_id",
            ],
        )

        manual_confirmed_col = first_existing_column(
            template_df,
            [
                "manual_confirmed",
                "confirmed_match",
                "final_match",
            ],
        )

        suggested_col = first_existing_column(
            template_df,
            [
                "suggested_match",
                "best_candidate",
                "nb04_recording_name",
            ],
        )

        if native_id_col is not None:
            for _, row in template_df.iterrows():
                native_id = clean_string_value(row.get(native_id_col))

                if native_id == "":
                    continue

                template_by_id[native_id] = {
                    "template_manual_confirmed": clean_string_value(
                        row.get(manual_confirmed_col)
                    )
                    if manual_confirmed_col
                    else "",
                    "template_suggested": clean_string_value(
                        row.get(suggested_col)
                    )
                    if suggested_col
                    else "",
                }

    rows = []

    for recording_id in native_inventory_df["recording_id"].tolist():
        manual_decision = MANUAL_NATIVE_TO_NB04_DECISIONS.get(recording_id)

        template_decision = template_by_id.get(
            recording_id,
            {
                "template_manual_confirmed": "",
                "template_suggested": "",
            },
        )

        template_manual_confirmed = template_decision.get("template_manual_confirmed", "")
        template_suggested = template_decision.get("template_suggested", "")

        if manual_decision is not None:
            nb04_name = manual_decision.get("nb04_recording_name", "")
            status = manual_decision.get("nb04_match_status", "unreviewed")
            reason = manual_decision.get("reason", "manual_decision_no_reason")

            source = "manual_validated_decision_dict"

            if template_manual_confirmed != "":
                source = "manual_validated_decision_dict_with_template_manual_column_present"
            elif template_suggested != "":
                source = "manual_validated_decision_dict_over_template_suggestion"

            if template_manual_confirmed != "" or template_suggested != "":
                template_status_note = (
                    f"template_manual_confirmed={template_manual_confirmed}; "
                    f"template_suggested={template_suggested}"
                )
            else:
                template_status_note = ""

        else:
            # Conservative fallback for future recordings not yet reviewed.
            if template_manual_confirmed != "" and template_manual_confirmed.upper() != "NO_MATCH":
                nb04_name = template_manual_confirmed
                status = "confirmed"
                source = "template_manual_confirmed_no_manual_dict_entry"
                reason = "manual_dict_missing_used_template_manual_confirmed"

            elif template_manual_confirmed.upper() == "NO_MATCH":
                nb04_name = ""
                status = "no_match"
                source = "template_manual_confirmed_no_manual_dict_entry"
                reason = "manual_dict_missing_template_manual_no_match"

            elif template_suggested.upper() == "NO_MATCH":
                nb04_name = ""
                status = "no_match"
                source = "template_suggested_no_manual_dict_entry"
                reason = "manual_dict_missing_template_suggested_no_match"

            elif "CONFLICT" in template_suggested.upper():
                nb04_name = ""
                status = "conflict"
                source = "template_suggested_no_manual_dict_entry"
                reason = f"manual_dict_missing_template_conflict: {template_suggested}"

            elif "AMBIG" in template_suggested.upper():
                nb04_name = ""
                status = "ambiguous"
                source = "template_suggested_no_manual_dict_entry"
                reason = f"manual_dict_missing_template_ambiguous: {template_suggested}"

            else:
                nb04_name = ""
                status = "unreviewed"
                source = "no_manual_decision_no_template_decision"
                reason = "needs_human_review"

            template_status_note = (
                f"template_manual_confirmed={template_manual_confirmed}; "
                f"template_suggested={template_suggested}"
            )

        rows.append(
            {
                "recording_id": recording_id,
                "nb04_recording_name": nb04_name,
                "nb04_match_status": status,
                "nb04_mapping_source": source,
                "nb04_mapping_reason": reason,
                "template_manual_confirmed": template_manual_confirmed,
                "template_suggested": template_suggested,
                "template_status_note": template_status_note,
            }
        )

    mapping_df = pd.DataFrame(rows)

    return mapping_df.sort_values("recording_id").reset_index(drop=True)

def build_nb04_registry_lookup(registry_csv_path):
    """
    Build lookup table from NB04 accepted recording registry.

    Parameters
    ----------
    registry_csv_path : pathlib.Path or str
        Path to NB04 registry CSV.

    Returns
    -------
    pandas.DataFrame
        Simplified registry table.
    """
    registry_df = load_optional_csv(registry_csv_path)

    if len(registry_df) == 0:
        return pd.DataFrame(
            columns=[
                "nb04_recording_name",
                "nb04_has_registry_row",
                "nb04_audio_path",
                "nb04_audio_path_exists",
                "nb04_video_path",
                "nb04_video_path_exists",
                "nb04_registry_audio_duration_s",
            ]
        )

    recording_col = first_existing_column(registry_df, ["recording_name", "nb04_recording_name"])

    if recording_col is None:
        raise ValueError("NB04 registry CSV has no recording_name column.")

    audio_path_col = first_existing_column(registry_df, ["audio_path", "audio_wav_path", "wav_path"])
    video_path_col = first_existing_column(registry_df, ["video_path", "avi_path", "recording_path"])
    audio_duration_col = first_existing_column(registry_df, ["audio_duration_s", "audio_duration_s_x", "audio_duration_s_y",  "duration_audio_s"])

    rows = []

    for _, row in registry_df.iterrows():
        nb04_name = clean_string_value(row.get(recording_col))

        if nb04_name == "":
            continue

        audio_path = clean_string_value(row.get(audio_path_col)) if audio_path_col else ""
        video_path = clean_string_value(row.get(video_path_col)) if video_path_col else ""
        audio_duration = None

        if audio_duration_col:
            try:
                audio_duration = float(row.get(audio_duration_col))
            except Exception:
                audio_duration = None

        rows.append(
            {
                "nb04_recording_name": nb04_name,
                "nb04_has_registry_row": True,
                "nb04_audio_path": audio_path,
                "nb04_audio_path_exists": Path(audio_path).exists() if audio_path else False,
                "nb04_video_path": video_path,
                "nb04_video_path_exists": Path(video_path).exists() if video_path else False,
                "nb04_registry_audio_duration_s": audio_duration,
            }
        )

    return pd.DataFrame(rows)


def build_nb04_morphology_peak_summary(morphology_csv_path):
    """
    Build NB04 image peak availability summary from morphology candidates CSV.

    Parameters
    ----------
    morphology_csv_path : pathlib.Path or str
        Path to NB04 morphology candidates CSV.

    Returns
    -------
    pandas.DataFrame
        One row per NB04 recording.
    """

    morphology_df = load_optional_csv(morphology_csv_path)

    if len(morphology_df) == 0:
        return pd.DataFrame(
            columns=[
                "nb04_recording_name",
                "nb04_morphology_rows",
                "nb04_complete_beat_rows",
                "nb04_has_image_peaks",
                "nb04_first_image_peak_s",
                "nb04_last_image_peak_s",
            ]
        )
    recording_col = first_existing_column(morphology_df, ["recording_name", "nb04_recording_name"])

    if recording_col is None:
        raise ValueError("NB04 morphology CSV has no recording_name column.")

    complete_col = first_existing_column(morphology_df, ["is_complete_beat", "complete_beat", "is_complete"])
    time_col = first_existing_column(morphology_df, ["video_frame_time_s", "beat_time_s", "peak_time_s", "time_s"])
    rows = []

    for nb04_name, group_df in morphology_df.groupby(recording_col):
        nb04_name = clean_string_value(nb04_name)

        if complete_col is not None:
            complete_mask = group_df[complete_col].astype(str).str.lower().isin(["true", "1", "yes"])
            complete_df = group_df[complete_mask].copy()
        else:
            complete_df = group_df.copy()

        peak_times = []

        if time_col is not None:
            peak_times = pd.to_numeric(complete_df[time_col], errors="coerce").dropna().tolist()

        rows.append(
            {
                "nb04_recording_name": nb04_name,
                "nb04_morphology_rows": int(len(group_df)),
                "nb04_complete_beat_rows": int(len(complete_df)),
                "nb04_has_image_peaks": len(peak_times) > 0,
                "nb04_first_image_peak_s": round(float(min(peak_times)), 4)
                if len(peak_times) > 0 else None,
                "nb04_last_image_peak_s": round(float(max(peak_times)), 4)
                if len(peak_times) > 0 else None,
            }
        )

    return pd.DataFrame(rows)


def build_native_nb04_mapping_availability_table(
    native_inventory_df,
    template_csv_path,
    registry_csv_path,
    morphology_csv_path,
):
    """
    Build native-to-NB04 mapping and NB04 availability table.

    Parameters
    ----------
    native_inventory_df : pandas.DataFrame
        Native inventory table.

    template_csv_path : pathlib.Path
        Manual mapping template CSV.

    registry_csv_path : pathlib.Path
        NB04 accepted recording registry CSV.

    morphology_csv_path : pathlib.Path
        NB04 morphology candidates CSV.

    Returns
    -------
    pandas.DataFrame
        One row per native recording.
    """
    mapping_df = build_mapping_from_template_csv(template_csv_path=template_csv_path, native_inventory_df=native_inventory_df,)
    registry_lookup_df = build_nb04_registry_lookup(registry_csv_path)
    morphology_summary_df = build_nb04_morphology_peak_summary(morphology_csv_path)

    out_df = mapping_df.merge(registry_lookup_df, how="left", on="nb04_recording_name")
    out_df = out_df.merge(morphology_summary_df, how="left", on="nb04_recording_name")
    boolean_fill_cols = [
        "nb04_has_registry_row",
        "nb04_audio_path_exists",
        "nb04_video_path_exists",
        "nb04_has_image_peaks",
    ]

    for col in boolean_fill_cols:
        if col in out_df.columns:
            out_df[col] = out_df[col].fillna(False).astype(bool)

    numeric_fill_cols = ["nb04_morphology_rows", "nb04_complete_beat_rows",]

    for col in numeric_fill_cols:
        if col in out_df.columns:
            out_df[col] = out_df[col].fillna(0).astype(int)

    if "nb04_audio_path" in out_df.columns:
        out_df["nb04_audio_path"] = out_df["nb04_audio_path"].fillna("")

    if "nb04_video_path" in out_df.columns:
        out_df["nb04_video_path"] = out_df["nb04_video_path"].fillna("")

    if "nb04_registry_audio_duration_s" in out_df.columns:
        out_df["nb04_registry_audio_duration_s"] = pd.to_numeric(
            out_df["nb04_registry_audio_duration_s"],
            errors="coerce",
        )

    return out_df.sort_values("recording_id").reset_index(drop=True)


def summarize_native_nb04_mapping(mapping_availability_df):
    """
    Print compact summary of native-to-NB04 mapping availability.

    Parameters
    ----------
    mapping_availability_df : pandas.DataFrame
        Mapping availability table.
    """
    print("Native-to-NB04 mapping and availability")
    print("=" * 80)
    print(f"Rows: {len(mapping_availability_df)}")
    print()
    print("nb04_match_status counts:")
    print(mapping_availability_df["nb04_match_status"].value_counts(dropna=False))
    print()

    if "nb04_audio_path_exists" in mapping_availability_df.columns:
        print(f"Audio path exists: {int(mapping_availability_df['nb04_audio_path_exists'].sum())}/{len(mapping_availability_df)}")
    if "nb04_has_image_peaks" in mapping_availability_df.columns:
        print(f"Image peaks available: {int(mapping_availability_df['nb04_has_image_peaks'].sum())}/{len(mapping_availability_df)}")

    print()

    summary_cols = [
        "recording_id",
        "nb04_recording_name",
        "nb04_match_status",
        "nb04_mapping_source",
        "nb04_mapping_reason",
        "nb04_audio_path_exists",
        "nb04_registry_audio_duration_s",
        "nb04_has_image_peaks",
        "nb04_complete_beat_rows",
        "nb04_first_image_peak_s",
        "nb04_last_image_peak_s",
    ]

    summary_cols = [col for col in summary_cols if col in mapping_availability_df.columns]
    display(mapping_availability_df[summary_cols])


def save_native_nb04_mapping_table(mapping_availability_df, output_dir):
    """
    Save native-to-NB04 mapping availability table.

    Parameters
    ----------
    mapping_availability_df : pandas.DataFrame
        Mapping table.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    pathlib.Path
        Saved CSV path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "nb06_v2_native_to_nb04_mapping_availability.csv"
    mapping_availability_df.to_csv(output_path, index=False)

    print(f"Saved native-to-NB04 mapping availability CSV: {output_path}")

    return output_path


def validate_expected_native_nb04_mapping(mapping_availability_df):
    """
    Validate expected NB06 V2 native-to-NB04 mapping.

    Expected for this batch:
    - 8 confirmed mappings,
    - 2 no_match recordings: 004SMP and 008SMP,
    - no ambiguous/conflict final statuses.

    Parameters
    ----------
    mapping_availability_df : pandas.DataFrame
        Native-to-NB04 mapping availability table.

    Returns
    -------
    pandas.DataFrame
        Problem rows. Empty means validation passed.
    """
    expected_no_match = {"202606130417060004SMP", "202606130430380008SMP",}
    problems = []

    for _, row in mapping_availability_df.iterrows():
        recording_id = row["recording_id"]
        status = row["nb04_match_status"]
        nb04_name = clean_string_value(row.get("nb04_recording_name", ""))

        if recording_id in expected_no_match:
            if status != "no_match" or nb04_name != "":
                problems.append(
                    {
                        "recording_id": recording_id,
                        "expected": "no_match",
                        "actual_status": status,
                        "actual_nb04_name": nb04_name,
                    }
                )
        else:
            if status != "confirmed" or nb04_name == "":
                problems.append(
                    {
                        "recording_id": recording_id,
                        "expected": "confirmed",
                        "actual_status": status,
                        "actual_nb04_name": nb04_name,
                    }
                )

    return pd.DataFrame(problems)

In [19]:
native_nb04_mapping_df = build_native_nb04_mapping_availability_table(
    native_inventory_df=native_inventory_df,
    template_csv_path=PATHS["NATIVE_TO_NB04_TEMPLATE_CSV"],
    registry_csv_path=PATHS["NB04_REGISTRY_CSV"],
    morphology_csv_path=PATHS["NB04_MORPHOLOGY_CSV"],
)
summarize_native_nb04_mapping(native_nb04_mapping_df)
native_nb04_mapping_csv_path = save_native_nb04_mapping_table(
    native_nb04_mapping_df,
    PATHS["NB06_V2_REPORTS_DIR"],
)
native_nb04_mapping_df


Native-to-NB04 mapping and availability
Rows: 10

nb04_match_status counts:
nb04_match_status
confirmed    8
no_match     2
Name: count, dtype: int64

Audio path exists: 8/10
Image peaks available: 6/10



,recording_id,nb04_recording_name,nb04_match_status,nb04_mapping_source,nb04_mapping_reason,nb04_audio_path_exists,nb04_registry_audio_duration_s,nb04_has_image_peaks,nb04_complete_beat_rows,nb04_first_image_peak_s,nb04_last_image_peak_s
0,202606130411060002SMP,candidate_test_02_brachial,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,True,59.242667,True,4,12.4667,43.2667
1,202606130413540003SMP,candidate_test_03_brachial,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,True,46.997333,True,4,8.1333,26.1000
2,202606130417060004SMP,,no_match,manual_validated_decision_dict_over_template_s...,no_valid_nb04_counterpart,False,NaN,False,0,NaN,NaN
3,202606130420260005SMP,candidate_test_08_brachial,confirmed,manual_validated_decision_dict_over_template_s...,confirmed_by_frame_count,True,45.546667,True,44,17.7667,38.9667
4,202606130422440006SMP,candidate_test_02_neck,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,True,51.264000,False,0,NaN,NaN
5,202606130426500007SMP,candidate_test_05_brachial,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,True,36.704000,True,55,4.1667,34.9667
6,202606130430380008SMP,,no_match,manual_validated_decision_dict_over_template_s...,duration_conflict_no_valid_nb04_counterpart,False,NaN,False,0,NaN,NaN
7,202606130433280009SMP,candidate_test_04_brachial_2,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,True,31.733333,True,6,10.6667,27.3667
8,202606130434130010SMP,candidate_test_04_brachial_1,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,True,33.845333,True,7,4.3333,32.3333
9,202606130437480012SMP,candidate_test_01_brachial,confirmed,manual_validated_decision_dict_over_template_s...,confirmed_by_frame_count,True,45.045333,False,0,NaN,NaN


Saved native-to-NB04 mapping availability CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_native_to_nb04_mapping_availability.csv


,recording_id,nb04_recording_name,nb04_match_status,nb04_mapping_source,nb04_mapping_reason,template_manual_confirmed,template_suggested,template_status_note,nb04_has_registry_row,nb04_audio_path,nb04_audio_path_exists,nb04_video_path,nb04_video_path_exists,nb04_registry_audio_duration_s,nb04_morphology_rows,nb04_complete_beat_rows,nb04_has_image_peaks,nb04_first_image_peak_s,nb04_last_image_peak_s
0,202606130411060002SMP,candidate_test_02_brachial,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,,candidate_test_02_brachial,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,59.242667,4,4,True,12.4667,43.2667
1,202606130413540003SMP,candidate_test_03_brachial,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,,candidate_test_03_brachial,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,46.997333,4,4,True,8.1333,26.1000
2,202606130417060004SMP,,no_match,manual_validated_decision_dict_over_template_s...,no_valid_nb04_counterpart,,NO_MATCH,template_manual_confirmed=; template_suggested...,False,,False,,False,NaN,0,0,False,NaN,NaN
3,202606130420260005SMP,candidate_test_08_brachial,confirmed,manual_validated_decision_dict_over_template_s...,confirmed_by_frame_count,,AMBIGUOUS_SEE_REPORT,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,45.546667,44,44,True,17.7667,38.9667
4,202606130422440006SMP,candidate_test_02_neck,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,,candidate_test_02_neck,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,51.264000,0,0,False,NaN,NaN
5,202606130426500007SMP,candidate_test_05_brachial,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,,candidate_test_05_brachial,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,36.704000,55,55,True,4.1667,34.9667
6,202606130430380008SMP,,no_match,manual_validated_decision_dict_over_template_s...,duration_conflict_no_valid_nb04_counterpart,,CONFLICT_SEE_REPORT,template_manual_confirmed=; template_suggested...,False,,False,,False,NaN,0,0,False,NaN,NaN
7,202606130433280009SMP,candidate_test_04_brachial_2,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,,candidate_test_04_brachial_2,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,31.733333,6,6,True,10.6667,27.3667
8,202606130434130010SMP,candidate_test_04_brachial_1,confirmed,manual_validated_decision_dict_over_template_s...,manual_validated_mapping,,candidate_test_04_brachial_1,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,33.845333,7,7,True,4.3333,32.3333
9,202606130437480012SMP,candidate_test_01_brachial,confirmed,manual_validated_decision_dict_over_template_s...,confirmed_by_frame_count,,AMBIGUOUS_SEE_REPORT,template_manual_confirmed=; template_suggested...,True,D:\code\DopplerLab\audio_exports\batch_2026_06...,True,D:\code\DopplerLab\ultrasound_recordings\batch...,True,45.045333,0,0,False,NaN,NaN


In [21]:
mapping_validation_problem_df = validate_expected_native_nb04_mapping(native_nb04_mapping_df)

print()
print("Mapping validation")
print("=" * 80)

if len(mapping_validation_problem_df) == 0:
    print("PASS — mapping matches expected NB06 V2 decisions.")
else:
    display(mapping_validation_problem_df)


Mapping validation
PASS — mapping matches expected NB06 V2 decisions.


## Final NB06 V2 recording diagnostic table

This section merges the validated NB06 V2 building blocks:

- native recording inventory,
- PW metadata calibration QC,
- native / AVI / audio duration QC,
- manual native-to-NB04 mapping,
- NB04 audio and image peak availability.

The output is a single recording-level diagnostic table.

Important decisions:

- Native individual beat timing remains `failed_validation`.
- Native HR/QC remains experimental and does not control `recommended_use`.
- Recordings are recommended based on confirmed metadata, timing, mapping, audio/image availability, and project scope.

In [22]:
def assign_nb06_v2_recommended_use(row):
    """
    Assign final NB06 V2 recommended_use label.

    This function intentionally does NOT use native HR as a decision gate.
    Native HR/QC remains experimental after Scope 01.

    Labels
    ------
    use_for_nb07_demo
        Confirmed NB04 match, metadata pass, timing pass, audio available,
        and image peaks available.

    use_for_nb06_qc
        Native recording is technically valid and useful for diagnostic/QC work,
        but not suitable for NB07 demo because image peaks are missing
        or no confirmed NB04 image counterpart exists.

    native_only_no_image_match
        Native recording has no confirmed NB04 counterpart.
        Useful for native metadata/timing documentation only.

    reject_or_review
        Serious conflict, missing metadata, failed timing, or unreviewed mapping.

    documentation_only
        Kept for completeness but not used for active analysis.

    Parameters
    ----------
    row : pandas.Series
        One merged diagnostic row.

    Returns
    -------
    tuple[str, str]
        recommended_use, recommended_use_reason
    """
    recording_id = row.get("recording_id", "")
    nb04_status = row.get("nb04_match_status", "")
    metadata_qc = row.get("metadata_qc", "")
    timing_qc = row.get("timing_qc", "")

    has_pw_bin = bool(row.get("has_pw_bin", False))
    has_linked_avi = bool(row.get("has_linked_avi", False))
    has_audio = bool(row.get("nb04_audio_path_exists", False))
    has_image_peaks = bool(row.get("nb04_has_image_peaks", False))

    if not has_pw_bin:
        return ("reject_or_review", "missing_pw_binary",)
    if metadata_qc != "pass":
        return ("reject_or_review", f"metadata_qc={metadata_qc}",)
    if timing_qc != "pass":
        return ("reject_or_review", f"timing_qc={timing_qc}",)
    if nb04_status in ("ambiguous", "conflict", "unreviewed"):
        return ("reject_or_review", f"nb04_match_status={nb04_status}",)

    if nb04_status == "no_match":
        return ("native_only_no_image_match", "no_confirmed_nb04_counterpart; native metadata/timing only",)

    if nb04_status == "confirmed":
        if has_audio and has_image_peaks and has_linked_avi:
            return ("use_for_nb07_demo", "confirmed_nb04_match; audio_available; image_peaks_available; metadata_and_timing_pass")
        if has_audio and not has_image_peaks:
            return ("use_for_nb06_qc", "confirmed_nb04_match_and_audio_available_but_no_nb04_image_peaks",)

        if not has_audio:
            return ("documentation_only", "confirmed_nb04_match_but_audio_missing",)

    return ("documentation_only", f"fallback_state_recording_id={recording_id}",)


def build_nb06_v2_final_diagnostic_table(
    native_inventory_df,
    pw_calibration_qc_df,
    duration_qc_df,
    native_nb04_mapping_df,
):
    """
    Build final NB06 V2 recording-level diagnostic table.

    Parameters
    ----------
    native_inventory_df : pandas.DataFrame
        Native recording inventory.

    pw_calibration_qc_df : pandas.DataFrame
        PW metadata calibration QC table.

    duration_qc_df : pandas.DataFrame
        Duration agreement QC table.

    native_nb04_mapping_df : pandas.DataFrame
        Native-to-NB04 mapping and availability table.

    Returns
    -------
    pandas.DataFrame
        Final NB06 V2 recording diagnostic table.
    """
    # Start from inventory because it has one row per native recording.
    final_df = native_inventory_df.copy()

    calibration_keep_cols = [
        "recording_id",
        "metadata_qc",
        "metadata_qc_reason",
        "pw_region_name",
        "pw_roi_x0",
        "pw_roi_x1",
        "pw_roi_y0",
        "pw_roi_y1",
        "pw_roi_width_px",
        "pw_roi_height_px",
        "pw_baseline_y_roi_px",
        "pw_baseline_y_global_px",
        "pw_phy_delta_x_s_per_px",
        "pw_phy_delta_y_cm_s_per_px",
        "display_pages_per_pixel",
        "velocity_formula",
    ]

    calibration_keep_cols = [col for col in calibration_keep_cols if col in pw_calibration_qc_df.columns]
    final_df = final_df.merge(pw_calibration_qc_df[calibration_keep_cols], how="left",on="recording_id",)
    duration_keep_cols = [
        "recording_id",
        "duration_avi_s",
        "duration_audio_s",
        "duration_format_s",
        "video_frame_count",
        "video_fps",
        "media_duration_source",
        "pages_vs_avi_s",
        "pages_vs_audio_s",
        "max_primary_discrepancy_s",
        "timing_qc",
        "timing_qc_reason",
        "app_xml_qc",
    ]

    duration_keep_cols = [col for col in duration_keep_cols if col in duration_qc_df.columns]
    final_df = final_df.merge(duration_qc_df[duration_keep_cols], how="left", on="recording_id",)

    mapping_keep_cols = [
        "recording_id",
        "nb04_recording_name",
        "nb04_match_status",
        "nb04_mapping_source",
        "nb04_mapping_reason",
        "template_manual_confirmed",
        "template_suggested",
        "template_status_note",
        "nb04_has_registry_row",
        "nb04_audio_path",
        "nb04_audio_path_exists",
        "nb04_video_path",
        "nb04_video_path_exists",
        "nb04_registry_audio_duration_s",
        "nb04_morphology_rows",
        "nb04_complete_beat_rows",
        "nb04_has_image_peaks",
        "nb04_first_image_peak_s",
        "nb04_last_image_peak_s",
    ]

    mapping_keep_cols = [col for col in mapping_keep_cols if col in native_nb04_mapping_df.columns]
    final_df = final_df.merge(native_nb04_mapping_df[mapping_keep_cols], how="left", on="recording_id",)
    # Stable decision/status columns.
    final_df["native_beat_timing_status"] = NATIVE_BEAT_TIMING_STATUS
    final_df["native_hr_qc_status"] = "experimental_not_decisional"
    final_df["native_hr_qc_reason"] = ("Scope01 suggests coarse native HR/QC may be possible, but no production-ready signal was selected.")
    final_df["field_a_status"] = "codebook_like_documentation_only"
    final_df["bmode_tracking_status"] = "not_integrated_scope02_not_visible"
    final_df["physio_status"] = "referenced_no_persisted_stream"
    final_df["feparam_status"] = "future_metadata_extraction_candidate"
    final_df["bc_partition_status"] = "future_static_blob_exploration_candidate"
    
    recommended = final_df.apply(assign_nb06_v2_recommended_use, axis=1,)
    final_df["recommended_use"] = [item[0] for item in recommended]
    final_df["recommended_use_reason"] = [item[1] for item in recommended]

    # Clean boolean columns after merge.
    bool_cols = [
        "has_pw_bin",
        "has_linked_avi",
        "has_dcm_region_para",
        "nb04_audio_path_exists",
        "nb04_video_path_exists",
        "nb04_has_image_peaks",
    ]

    for col in bool_cols:
        if col in final_df.columns:
            final_df[col] = final_df[col].fillna(False).astype(bool)

    # Final column order.
    preferred_cols = [
        "recording_id",

        # Final decision
        "recommended_use",
        "recommended_use_reason",

        # Core availability
        "has_pw_bin",
        "has_linked_avi",
        "has_dcm_region_para",

        # Metadata QC
        "metadata_qc",
        "metadata_qc_reason",
        "pw_region_name",
        "pw_roi_x0",
        "pw_roi_x1",
        "pw_roi_y0",
        "pw_roi_y1",
        "pw_baseline_y_global_px",
        "pw_phy_delta_x_s_per_px",
        "pw_phy_delta_y_cm_s_per_px",
        "display_pages_per_pixel",

        # Timing QC
        "duration_pages_s",
        "duration_avi_s",
        "duration_audio_s",
        "video_frame_count",
        "video_fps",
        "pages_vs_avi_s",
        "pages_vs_audio_s",
        "max_primary_discrepancy_s",
        "timing_qc",
        "timing_qc_reason",

        # Mapping / NB04
        "nb04_recording_name",
        "nb04_match_status",
        "nb04_mapping_source",
        "nb04_mapping_reason",
        "template_suggested",
        "nb04_audio_path_exists",
        "nb04_registry_audio_duration_s",
        "nb04_has_image_peaks",
        "nb04_complete_beat_rows",
        "nb04_first_image_peak_s",
        "nb04_last_image_peak_s",

        # Explicit negative / experimental statuses
        "native_beat_timing_status",
        "native_hr_qc_status",
        "native_hr_qc_reason",
        "field_a_status",
        "bmode_tracking_status",
        "physio_status",
        "feparam_status",
        "bc_partition_status",

        # Paths
        "recording_folder",
        "pw_bin_path",
        "linked_avi_path",
        "dcm_region_para_path",
        "nb04_audio_path",
        "nb04_video_path",
    ]

    preferred_cols = [col for col in preferred_cols if col in final_df.columns]
    remaining_cols = [col for col in final_df.columns if col not in preferred_cols]
    final_df = final_df[preferred_cols + remaining_cols]

    return final_df.sort_values("recording_id").reset_index(drop=True)


def summarize_nb06_v2_final_diagnostic(final_df):
    """
    Print compact summary of final NB06 V2 diagnostic table.

    Parameters
    ----------
    final_df : pandas.DataFrame
        Final diagnostic table.
    """
    print("NB06 V2 final recording diagnostic table")
    print("=" * 80)
    print(f"Rows: {len(final_df)}")
    print()
    print("recommended_use counts:")
    print(final_df["recommended_use"].value_counts(dropna=False))
    print()
    print("metadata_qc counts:")
    print(final_df["metadata_qc"].value_counts(dropna=False))
    print()
    print("timing_qc counts:")
    print(final_df["timing_qc"].value_counts(dropna=False))
    print()
    print("nb04_match_status counts:")
    print(final_df["nb04_match_status"].value_counts(dropna=False))
    print()

    summary_cols = [
        "recording_id",
        "recommended_use",
        "nb04_recording_name",
        "nb04_match_status",
        "metadata_qc",
        "timing_qc",
        "nb04_audio_path_exists",
        "nb04_has_image_peaks",
        "nb04_complete_beat_rows",
        "native_beat_timing_status",
        "native_hr_qc_status",
    ]

    summary_cols = [col for col in summary_cols if col in final_df.columns]
    display(final_df[summary_cols])


def save_nb06_v2_final_diagnostic(final_df, output_dir):
    """
    Save final NB06 V2 diagnostic table.

    Parameters
    ----------
    final_df : pandas.DataFrame
        Final diagnostic table.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    pathlib.Path
        Saved CSV path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "nb06_v2_recording_diagnostic.csv"
    final_df.to_csv(output_path, index=False)
    print(f"Saved final NB06 V2 diagnostic CSV: {output_path}")

    return output_path


In [23]:
nb06_v2_diagnostic_df = build_nb06_v2_final_diagnostic_table(
    native_inventory_df=native_inventory_df,
    pw_calibration_qc_df=pw_calibration_qc_df,
    duration_qc_df=duration_qc_df,
    native_nb04_mapping_df=native_nb04_mapping_df,
)
summarize_nb06_v2_final_diagnostic(nb06_v2_diagnostic_df)
nb06_v2_diagnostic_csv_path = save_nb06_v2_final_diagnostic(nb06_v2_diagnostic_df, PATHS["NB06_V2_REPORTS_DIR"],)
nb06_v2_diagnostic_df

NB06 V2 final recording diagnostic table
Rows: 10

recommended_use counts:
recommended_use
use_for_nb07_demo             6
native_only_no_image_match    2
use_for_nb06_qc               2
Name: count, dtype: int64

metadata_qc counts:
metadata_qc
pass    10
Name: count, dtype: int64

timing_qc counts:
timing_qc
pass    10
Name: count, dtype: int64

nb04_match_status counts:
nb04_match_status
confirmed    8
no_match     2
Name: count, dtype: int64



,recording_id,recommended_use,nb04_recording_name,nb04_match_status,metadata_qc,timing_qc,nb04_audio_path_exists,nb04_has_image_peaks,nb04_complete_beat_rows,native_beat_timing_status,native_hr_qc_status
0,202606130411060002SMP,use_for_nb07_demo,candidate_test_02_brachial,confirmed,pass,pass,True,True,4,failed_validation,experimental_not_decisional
1,202606130413540003SMP,use_for_nb07_demo,candidate_test_03_brachial,confirmed,pass,pass,True,True,4,failed_validation,experimental_not_decisional
2,202606130417060004SMP,native_only_no_image_match,,no_match,pass,pass,False,False,0,failed_validation,experimental_not_decisional
3,202606130420260005SMP,use_for_nb07_demo,candidate_test_08_brachial,confirmed,pass,pass,True,True,44,failed_validation,experimental_not_decisional
4,202606130422440006SMP,use_for_nb06_qc,candidate_test_02_neck,confirmed,pass,pass,True,False,0,failed_validation,experimental_not_decisional
5,202606130426500007SMP,use_for_nb07_demo,candidate_test_05_brachial,confirmed,pass,pass,True,True,55,failed_validation,experimental_not_decisional
6,202606130430380008SMP,native_only_no_image_match,,no_match,pass,pass,False,False,0,failed_validation,experimental_not_decisional
7,202606130433280009SMP,use_for_nb07_demo,candidate_test_04_brachial_2,confirmed,pass,pass,True,True,6,failed_validation,experimental_not_decisional
8,202606130434130010SMP,use_for_nb07_demo,candidate_test_04_brachial_1,confirmed,pass,pass,True,True,7,failed_validation,experimental_not_decisional
9,202606130437480012SMP,use_for_nb06_qc,candidate_test_01_brachial,confirmed,pass,pass,True,False,0,failed_validation,experimental_not_decisional


Saved final NB06 V2 diagnostic CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_recording_diagnostic.csv


,recording_id,recommended_use,recommended_use_reason,has_pw_bin,has_linked_avi,has_dcm_region_para,metadata_qc,metadata_qc_reason,pw_region_name,pw_roi_x0,...,pw_baseline_y_roi_px,velocity_formula,duration_format_s,media_duration_source,app_xml_qc,template_manual_confirmed,template_status_note,nb04_has_registry_row,nb04_video_path_exists,nb04_morphology_rows
0,202606130411060002SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,59.2667,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,4
1,202606130413540003SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,47.0333,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,4
2,202606130417060004SMP,native_only_no_image_match,no_confirmed_nb04_counterpart; native metadata...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,42.5000,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,False,False,0
3,202606130420260005SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,45.5667,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,44
4,202606130422440006SMP,use_for_nb06_qc,confirmed_nb04_match_and_audio_available_but_n...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,51.3000,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,0
5,202606130426500007SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,36.7333,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,55
6,202606130430380008SMP,native_only_no_image_match,no_confirmed_nb04_counterpart; native metadata...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion0,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,44.4333,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,False,False,0
7,202606130433280009SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion0,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,31.7667,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,6
8,202606130434130010SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,33.8667,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,7
9,202606130437480012SMP,use_for_nb06_qc,confirmed_nb04_match_and_audio_available_but_n...,True,True,True,pass,pw_region_and_calibration_plausible,DcmRegion1,79,...,131,v_cm_s = (y_full_px - 376) * (-0.1769457646367...,45.0667,ffprobe_nb_frames_over_fps,unavailable,,template_manual_confirmed=; template_suggested...,True,True,0


## Final validation and compact control table

This section performs final validation of the NB06 V2 diagnostic table and exports a compact control table.

The full diagnostic table keeps detailed metadata and paths.

The compact control table is intended for quick project decisions:

- which recordings can be used for NB07 demo,
- which are QC-only,
- which are native-only documentation cases.

In [24]:
def validate_nb06_v2_diagnostic_table(diagnostic_df):
    """
    Validate the final NB06 V2 diagnostic table.

    This validation encodes the expected state for the current
    batch_2026_06_13_native dataset.

    Parameters
    ----------
    diagnostic_df : pandas.DataFrame
        Final NB06 V2 diagnostic table.

    Returns
    -------
    pandas.DataFrame
        Validation checks. One row per check.
    """
    checks = []

    def add_check(check_name, passed, observed, expected, severity="error"):
        checks.append(
            {
                "check_name": check_name,
                "passed": bool(passed),
                "observed": observed,
                "expected": expected,
                "severity": severity,
            }
        )
    n_rows = len(diagnostic_df)

    add_check(check_name="row_count", passed=n_rows == 10, observed=n_rows, expected=10,)
    metadata_pass_count = int((diagnostic_df["metadata_qc"] == "pass").sum())
    add_check(check_name="metadata_qc_all_pass", passed=metadata_pass_count == 10, observed=metadata_pass_count, expected="10 pass",)
    timing_pass_count = int((diagnostic_df["timing_qc"] == "pass").sum())
    add_check(check_name="timing_qc_all_pass", passed=timing_pass_count == 10, observed=timing_pass_count, expected="10 pass",)
    match_counts = diagnostic_df["nb04_match_status"].value_counts().to_dict()
    add_check(check_name="nb04_confirmed_count", passed=match_counts.get("confirmed", 0) == 8, observed=match_counts.get("confirmed", 0), expected=8)
    add_check(check_name="nb04_no_match_count", passed=match_counts.get("no_match", 0) == 2, observed=match_counts.get("no_match", 0), expected=2,)
    bad_mapping_count = int(diagnostic_df["nb04_match_status"].isin(["ambiguous", "conflict", "unreviewed"]).sum())
    add_check(check_name="no_bad_final_mapping_status", passed=bad_mapping_count == 0, observed=bad_mapping_count, expected=0,)
    recommended_counts = diagnostic_df["recommended_use"].value_counts().to_dict()
    add_check(check_name="nb07_demo_count", passed=recommended_counts.get("use_for_nb07_demo", 0) == 6, 
              observed=recommended_counts.get("use_for_nb07_demo", 0), expected=6)
    add_check(check_name="nb06_qc_count", passed=recommended_counts.get("use_for_nb06_qc", 0) == 2, 
              observed=recommended_counts.get("use_for_nb06_qc", 0), expected=2)
    add_check(check_name="native_only_no_image_match_count", passed=recommended_counts.get("native_only_no_image_match", 0) == 2,
        observed=recommended_counts.get("native_only_no_image_match", 0), expected=2)
    reject_or_review_count = int((diagnostic_df["recommended_use"] == "reject_or_review").sum())
    add_check(check_name="no_reject_or_review_rows", passed=reject_or_review_count == 0, observed=reject_or_review_count, expected=0,)
    native_beat_failed_count = int((diagnostic_df["native_beat_timing_status"] == "failed_validation").sum())
    add_check(check_name="native_beat_timing_all_failed_validation", passed=native_beat_failed_count == 10,
        observed=native_beat_failed_count,expected=10,)
    native_hr_experimental_count = int((diagnostic_df["native_hr_qc_status"] == "experimental_not_decisional").sum())
    add_check(check_name="native_hr_all_experimental_not_decisional", passed=native_hr_experimental_count == 10,
        observed=native_hr_experimental_count,expected=10)
    nb07_df = diagnostic_df[diagnostic_df["recommended_use"] == "use_for_nb07_demo"]
    nb07_audio_and_image_count = int((nb07_df["nb04_audio_path_exists"].astype(bool) & nb07_df["nb04_has_image_peaks"].astype(bool)).sum())
    add_check(check_name="nb07_rows_have_audio_and_image_peaks", passed=nb07_audio_and_image_count == len(nb07_df),
        observed=nb07_audio_and_image_count, expected=len(nb07_df),)
    validation_df = pd.DataFrame(checks)

    return validation_df


def build_nb06_v2_compact_control_table(diagnostic_df):
    """
    Build a compact NB06 V2 control table.

    Parameters
    ----------
    diagnostic_df : pandas.DataFrame
        Final NB06 V2 diagnostic table.

    Returns
    -------
    pandas.DataFrame
        Compact decision table.
    """
    compact_cols = [
        "recording_id",
        "recommended_use",
        "recommended_use_reason",
        "nb04_recording_name",
        "nb04_match_status",
        "metadata_qc",
        "timing_qc",
        "duration_pages_s",
        "duration_avi_s",
        "duration_audio_s",
        "nb04_audio_path_exists",
        "nb04_has_image_peaks",
        "nb04_complete_beat_rows",
        "pw_roi_x0",
        "pw_roi_x1",
        "pw_roi_y0",
        "pw_roi_y1",
        "pw_baseline_y_global_px",
        "pw_phy_delta_x_s_per_px",
        "pw_phy_delta_y_cm_s_per_px",
        "display_pages_per_pixel",
        "native_beat_timing_status",
        "native_hr_qc_status",
    ]

    compact_cols = [col for col in compact_cols if col in diagnostic_df.columns]
    compact_df = diagnostic_df[compact_cols].copy()
    compact_df = compact_df.sort_values(["recommended_use", "recording_id"]).reset_index(drop=True)

    return compact_df


def write_nb06_v2_summary_markdown(diagnostic_df, validation_df, output_path):
    """
    Write a compact NB06 V2 summary markdown report.

    Parameters
    ----------
    diagnostic_df : pandas.DataFrame
        Final diagnostic table.

    validation_df : pandas.DataFrame
        Validation table.

    output_path : pathlib.Path or str
        Markdown output path.

    Returns
    -------
    pathlib.Path
        Saved markdown path.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    recommended_counts = diagnostic_df["recommended_use"].value_counts().to_dict()
    validation_pass = bool(validation_df["passed"].all())

    nb07_rows = diagnostic_df[diagnostic_df["recommended_use"] == "use_for_nb07_demo"]
    qc_rows = diagnostic_df[diagnostic_df["recommended_use"] == "use_for_nb06_qc"]
    native_only_rows = diagnostic_df[diagnostic_df["recommended_use"] == "native_only_no_image_match"]
    lines = []

    lines.append("# NB06 V2 Summary")
    lines.append("")
    lines.append("## Status")
    lines.append("")
    lines.append(f"- Validation pass: `{validation_pass}`")
    lines.append(f"- Total recordings: `{len(diagnostic_df)}`")
    lines.append("- Metadata QC: `10/10 pass`")
    lines.append("- Timing QC: `10/10 pass`")
    lines.append("- Native individual beat timing: `failed_validation`")
    lines.append("- Native HR/QC: `experimental_not_decisional`")
    lines.append("")
    lines.append("## Recommended use counts")
    lines.append("")

    for label, count in recommended_counts.items():
        lines.append(f"- `{label}`: `{count}`")

    lines.append("")
    lines.append("## NB07 demo candidates")
    lines.append("")

    for _, row in nb07_rows.iterrows():
        lines.append(f"- `{row['recording_id']}` → `{row['nb04_recording_name']}` ({int(row['nb04_complete_beat_rows'])} complete beat rows)")

    lines.append("")
    lines.append("## NB06 QC-only recordings")
    lines.append("")

    for _, row in qc_rows.iterrows():
        lines.append(f"- `{row['recording_id']}` → `{row['nb04_recording_name']}` reason: {row['recommended_use_reason']}")

    lines.append("")
    lines.append("## Native-only no-image-match recordings")
    lines.append("")

    for _, row in native_only_rows.iterrows():
        lines.append(f"- `{row['recording_id']}` reason: {row['recommended_use_reason']}")

    lines.append("")
    lines.append("## Confirmed calibration")
    lines.append("")
    lines.append("- PW ROI: `x=79–659`, `y=245–508`")
    lines.append("- Baseline: `y=376` full-frame pixels")
    lines.append("- Time scale: `0.006 s/pixel`")
    lines.append("- Velocity scale: `-0.1769457646 cm/s/pixel`")
    lines.append("- Display scale: `3.0 native pages/display pixel`")
    lines.append("")
    lines.append("## Important exclusions")
    lines.append("")
    lines.append("- No native velocity envelope.")
    lines.append("- No native PSV/EDV/RI/PI/VTI.")
    lines.append("- No native BP/SV/CO.")
    lines.append("- No native pair-range individual beat timing.")
    lines.append("- No field_a decoder.")
    lines.append("- No PHYSIO reader.")
    lines.append("- No FeParam clinical interpretation.")
    lines.append("- No B/BC diameter or stiffness tracking.")

    output_path.write_text("\n".join(lines), encoding="utf-8",)

    return output_path


def save_nb06_v2_validation_and_compact_outputs(validation_df, compact_df, diagnostic_df, paths):
    """
    Save final validation, compact table, and summary markdown.

    Parameters
    ----------
    validation_df : pandas.DataFrame
        Validation checks.

    compact_df : pandas.DataFrame
        Compact control table.

    diagnostic_df : pandas.DataFrame
        Final diagnostic table.

    paths : dict
        NB06 V2 path dictionary.

    Returns
    -------
    dict
        Saved output paths.
    """
    reports_dir = Path(paths["NB06_V2_REPORTS_DIR"])
    docs_dir = Path(paths["NB06_V2_DOCS_DIR"])
    reports_dir.mkdir(parents=True, exist_ok=True)
    docs_dir.mkdir(parents=True, exist_ok=True)
    validation_path = reports_dir / "nb06_v2_validation_checks.csv"
    compact_path = reports_dir / "nb06_v2_compact_control_table.csv"
    summary_path = docs_dir / "nb06_v2_summary.md"
    validation_df.to_csv(validation_path, index=False)
    compact_df.to_csv(compact_path, index=False)
    write_nb06_v2_summary_markdown(diagnostic_df=diagnostic_df, validation_df=validation_df,output_path=summary_path,)

    print(f"Saved validation checks CSV: {validation_path}")
    print(f"Saved compact control table CSV: {compact_path}")
    print(f"Saved summary markdown: {summary_path}")

    return {
        "validation_csv": validation_path,
        "compact_control_csv": compact_path,
        "summary_markdown": summary_path,
    }

In [25]:
nb06_v2_validation_df = validate_nb06_v2_diagnostic_table(nb06_v2_diagnostic_df)
nb06_v2_compact_control_df = build_nb06_v2_compact_control_table(nb06_v2_diagnostic_df)

print("NB06 V2 validation")
print("=" * 80)

if nb06_v2_validation_df["passed"].all():
    print("PASS — all validation checks passed.")
else:
    print("WARNING — at least one validation check failed.")
    display(nb06_v2_validation_df[~nb06_v2_validation_df["passed"]])

print()
display(nb06_v2_validation_df)

print()
print("Compact control table")
print("=" * 80)
display(nb06_v2_compact_control_df)

nb06_v2_final_output_paths = save_nb06_v2_validation_and_compact_outputs(
    validation_df=nb06_v2_validation_df,
    compact_df=nb06_v2_compact_control_df,
    diagnostic_df=nb06_v2_diagnostic_df,
    paths=PATHS,
)

NB06 V2 validation
PASS — all validation checks passed.



,check_name,passed,observed,expected,severity
0,row_count,True,10,10,error
1,metadata_qc_all_pass,True,10,10 pass,error
2,timing_qc_all_pass,True,10,10 pass,error
3,nb04_confirmed_count,True,8,8,error
4,nb04_no_match_count,True,2,2,error
5,no_bad_final_mapping_status,True,0,0,error
6,nb07_demo_count,True,6,6,error
7,nb06_qc_count,True,2,2,error
8,native_only_no_image_match_count,True,2,2,error
9,no_reject_or_review_rows,True,0,0,error



Compact control table


,recording_id,recommended_use,recommended_use_reason,nb04_recording_name,nb04_match_status,metadata_qc,timing_qc,duration_pages_s,duration_avi_s,duration_audio_s,...,pw_roi_x0,pw_roi_x1,pw_roi_y0,pw_roi_y1,pw_baseline_y_global_px,pw_phy_delta_x_s_per_px,pw_phy_delta_y_cm_s_per_px,display_pages_per_pixel,native_beat_timing_status,native_hr_qc_status
0,202606130417060004SMP,native_only_no_image_match,no_confirmed_nb04_counterpart; native metadata...,,no_match,pass,pass,42.464,42.5000,42.4747,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
1,202606130430380008SMP,native_only_no_image_match,no_confirmed_nb04_counterpart; native metadata...,,no_match,pass,pass,44.372,44.4333,44.3840,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
2,202606130422440006SMP,use_for_nb06_qc,confirmed_nb04_match_and_audio_available_but_n...,candidate_test_02_neck,confirmed,pass,pass,51.252,51.3000,51.2640,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
3,202606130437480012SMP,use_for_nb06_qc,confirmed_nb04_match_and_audio_available_but_n...,candidate_test_01_brachial,confirmed,pass,pass,45.026,45.0667,45.0453,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
4,202606130411060002SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,candidate_test_02_brachial,confirmed,pass,pass,59.230,59.2667,59.2427,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
5,202606130413540003SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,candidate_test_03_brachial,confirmed,pass,pass,46.982,47.0333,46.9973,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
6,202606130420260005SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,candidate_test_08_brachial,confirmed,pass,pass,45.528,45.5667,45.5467,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
7,202606130426500007SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,candidate_test_05_brachial,confirmed,pass,pass,36.692,36.7333,36.7040,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
8,202606130433280009SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,candidate_test_04_brachial_2,confirmed,pass,pass,31.724,31.7667,31.7333,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional
9,202606130434130010SMP,use_for_nb07_demo,confirmed_nb04_match; audio_available; image_p...,candidate_test_04_brachial_1,confirmed,pass,pass,33.832,33.8667,33.8453,...,79,659,245,508,376,0.006,-0.176946,3.0,failed_validation,experimental_not_decisional


Saved validation checks CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_validation_checks.csv
Saved compact control table CSV: E:\DopplerLab\reports\nb06_v2\nb06_v2_compact_control_table.csv
Saved summary markdown: E:\DopplerLab\docs\nb06_v2\nb06_v2_summary.md


## Final checkpoint

NB06 V2 is frozen at the first clean diagnostic/control-table checkpoint.

Generated outputs:

- `reports/nb06_v2/nb06_v2_recording_diagnostic.csv`
- `reports/nb06_v2/nb06_v2_compact_control_table.csv`
- `reports/nb06_v2/nb06_v2_validation_checks.csv`
- `docs/nb06_v2/nb06_v2_summary.md`

Final status:

- Metadata QC: 10/10 pass
- Timing QC: 10/10 pass
- Native-to-NB04 mapping: 8 confirmed, 2 no match
- NB07 demo candidates: 6 recordings
- NB06 QC-only recordings: 2 recordings
- Native-only no-image-match recordings: 2 recordings
- Native individual beat timing: failed validation
- Native HR/QC: experimental, not decisional

No further native decoding is integrated in NB06 V2.